**Pipeline (per cycle)**:
1. **Stereo Bbox Detection + Action Planning** — Qwen3-VL-4B sees left/right eye images, outputs dual bboxes + DSL actions
2. **Stereo Triangulation** — dual bbox centers → ray-ray intersection → 3D world position
3. **Execution** — OSC_POSE controller executes action primitives in Robosuite
4. **Loop** — When action buffer empties, Qwen re-plans with updated 3D context

---
## 1. System Dependencies (EGL for headless MuJoCo)

In [1]:
import subprocess, os

missing = any(
    subprocess.run(f"dpkg -l {pkg}".split(), capture_output=True).returncode != 0
    for pkg in ["libegl1-mesa", "libegl1-mesa-dev", "libglib2.0-0"]
)
if missing:
    !sudo apt-get update -qq && sudo apt-get install -y -qq libegl1-mesa libegl1-mesa-dev libglib2.0-0
    print("System deps installed.")
else:
    print("System deps already present.")

os.environ["MUJOCO_GL"] = "egl"
os.environ["EGL_DEVICE_ID"] = "0"



W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 10.)
debconf: falling back to frontend: Readline
Selecting previously unselected package libglx-dev:amd64.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../0-libglx-dev_1.4.0-1_amd64.deb ...
Unpacking libglx-dev:amd64 (1.4.0-1) ...
Selecting previously unselected package libgl-dev:amd64.
Preparing to unpack .../1-libgl-dev_1.4.0-1_amd64.deb ...
Unpacking libgl-dev:amd64 (1.4.0-1) ...
Selecting previously unselected package libegl-dev:amd64.
Preparing to unpack .../2-libegl-dev_1.4.0-1_amd64.deb ...
Unpacking libegl-dev:amd64 (1.4.0-1) ...
Selectin

---
## 2. Python Packages

In [ ]:
# Install robosuite + MuJoCo + VLM dependencies
import subprocess, sys

def _pip_install(pkg_spec):
    """Run pip install and raise on failure."""
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + pkg_spec.split(),
        capture_output=True, text=True)
    if r.returncode != 0:
        print(f"pip install {pkg_spec} FAILED:")
        print(r.stderr[-500:])
        r.check_returncode()  # raise

_pip_install("mujoco>=3.3.0 robosuite>=1.5.2")
# Qwen3-VL needs transformers >= 4.57.0
_pip_install("transformers>=4.57.0 accelerate bitsandbytes sentencepiece einops timm qwen-vl-utils")
_pip_install("opencv-python-headless")
# imageio-ffmpeg bundles ffmpeg WITH libx264 (OpenCV lacks it -> VideoWriter fails on Kaggle).
_pip_install("imageio imageio-ffmpeg")

# Create dummy flash_attn (Qwen3-VL may check for it)
import os
_stub_dir = "/tmp/flash_attn"
os.makedirs(_stub_dir, exist_ok=True)
with open(f"{_stub_dir}/__init__.py", "w") as f:
    f.write('__version__ = "2.6.1"\n')
if "/tmp" not in sys.path:
    sys.path.insert(0, "/tmp")
import flash_attn
print(f"flash_attn stub OK (version {flash_attn.__version__})")

# Kaggle's import hooks (ImportHookFinder / GcpModuleFinder at the front of sys.meta_path) force a
# broken, internally-mixed numpy from dist-packages: its strings.py imports _center from a umath that
# lacks it -> "cannot import name '_center'", breaking robosuite's scipy import. The hooks do this
# REGARDLESS of what pip installed or what sys.path says. Fix: install a CLEAN, consistent numpy+scipy
# into a dir we own (/tmp/cleanpkgs), then import them with the hooks temporarily disabled so the
# standard PathFinder resolves them off sys.path[0]. numpy is pinned to 2.0 -- NOT newer -- because
# robosuite -> numba demands "NumPy 2.0 or less"; a clean 2.0.2 is internally consistent (its own
# strings.py never references _center, which only exists in 2.2+), so the _center error is gone AND
# numba is satisfied. After this import numpy/scipy are cached in sys.modules for the whole kernel
# session, so the hooks never intercept them again.
import os, shutil, importlib.machinery as _im
_clean = "/tmp/cleanpkgs"
shutil.rmtree(_clean, ignore_errors=True)  # wipe any stale install from a prior run
os.makedirs(_clean, exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--target", _clean,
                "numpy==2.0.2", "scipy==1.14.1"], check=True, capture_output=True, text=True)
while _clean in sys.path:
    sys.path.remove(_clean)
sys.path.insert(0, _clean)
# Disable the numpy-hijacking hooks for THIS import only, then restore them for other packages.
_orig_meta = sys.meta_path[:]
sys.meta_path = [_im.BuiltinImporter, _im.FrozenImporter, _im.PathFinder]
for _k in [k for k in list(sys.modules) if k == "numpy" or k.startswith("numpy.")]:
    del sys.modules[_k]
import numpy, scipy
sys.meta_path = _orig_meta
print(f"numpy {numpy.__version__} @ {numpy.__file__}; scipy {scipy.__version__}", flush=True)

# Verify key packages
import mujoco
import robosuite as suite
# Silence repetitive robosuite controller-config warnings (BASIC composite config defines
# parts the single-arm Panda lacks: left/torso/head/base/legs). The StreamHandler grabbed the
# real stderr at import, so _silence_stderr() in make_env cannot catch it -> set level directly.
import logging as _logging
_logging.getLogger("robosuite_logs").setLevel(_logging.ERROR)
import transformers
print(f"mujoco {mujoco.__version__}, robosuite {suite.__version__}, transformers {transformers.__version__}")
print("Python deps OK")

---
## 3. Imports & GPU Check

In [ ]:
import json, re, time, math, logging, os, sys, contextlib, warnings, gc, random
from dataclasses import dataclass
from typing import Optional

import numpy as np
print(f"numpy {np.__version__}")

import torch

import matplotlib
matplotlib.use("Agg")

from PIL import Image

from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    Owlv2Processor,
    Owlv2ForObjectDetection,
)

# ── Suppress warnings before robosuite import ──
warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.CRITICAL)

# Redirect sys.stderr (Python level) so robosuite's logger captures the null stream
_stderr_null = open(os.devnull, "w")
_old_stderr = sys.stderr
sys.stderr = _stderr_null
# Also redirect at FD level for any C-level stderr writes during import
_fd_null = os.open(os.devnull, os.O_WRONLY)
_fd_old = os.dup(2)
os.dup2(_fd_null, 2)
os.close(_fd_null)
try:
    import robosuite as suite
    from robosuite.controllers import load_composite_controller_config
finally:
    # Restore FD level
    os.dup2(_fd_old, 2)
    os.close(_fd_old)
    # Restore Python level
    sys.stderr = _old_stderr
    _stderr_null.close()
print(f"robosuite {suite.__version__}")

import mujoco
print(f"mujoco {mujoco.__version__}")

# ── Post-import robosuite logging suppression ──
for _name in list(logging.root.manager.loggerDict):
    if _name == "robosuite" or _name.startswith("robosuite."):
        _log = logging.getLogger(_name)
        _log.disabled = True
        _log.setLevel(logging.CRITICAL)
        _log.handlers.clear()
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

SEED = None

def set_seed(seed: int):
    global SEED
    SEED = seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    mem = getattr(p, "total_memory", getattr(p, "total_mem", 0))
    print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {mem/1e9:.1f} GB")
else:
    print("No GPU — expect slow performance.")

print("Imports OK")

---
## 4. Config & Stereo Camera Setup

In [ ]:
@dataclass
class Config:
    vlm_model_id: str = "Qwen/Qwen3-VL-4B-Instruct"
    use_4bit: bool = True
    env_name: str = "Lift"
    robot: str = "Panda"
    task_type: str = "pick_up"   # "pick_up" (lift cube & hold) | "pick_and_place" (grasp cube, place into bin, release)
    bbox_mode: str = "auto"   # bbox detection: "auto" (Qwen decides) | "always" | "never"
    # ── Stereo camera ──
    left_camera: str = "left_eye"       # injected into MuJoCo XML
    right_camera: str = "right_eye"     # injected into MuJoCo XML
    stereo_use_table_z: bool = False    # use full stereo 3D (XYZ from triangulation)
    # ── Detection ──
    detection_queries: list = None  # OWLv2 text queries (overridden by bbox_target)
    bbox_target: Optional[str] = None  # OWLv2 query text (None = Qwen regenerates it each re-check round)
    object_x_range: tuple = (-0.20, -0.05)  # cube X range (meters, more negative = closer to robot)
    object_y_range: tuple = (-0.10, 0.10)   # cube Y range (meters)
    objects: dict = None          # {name: (x,y)|"random"|{"pos":..,"color":..,<ctor-kw>..}}; None -> single red cube; first entry = task object
    camera_size: int = 384
    stereo_baseline: float = 0.50       # wider baseline improves depth accuracy (was 0.30)
    # ── Planning ──
    plan_length_n: int = 4
    lift_height: float = 0.15
    n_trials: int = 10
    max_steps: int = 150
    seed: int = 42

    def __post_init__(self):
        self.camera_names = [self.left_camera, self.right_camera]
        if self.detection_queries is None:
            self.detection_queries = [self.bbox_target] if self.bbox_target else ["object"]
        if self.objects is None:
            self.objects = {"cube": {"pos": "random", "color": "red"}}

cfg = Config()
set_seed(cfg.seed)
print(f"VLM: {cfg.vlm_model_id}")
print(f"Camera: {cfg.camera_size}x{cfg.camera_size}, stereo baseline={cfg.stereo_baseline}m")
print(f"Plan: {cfg.plan_length_n} actions, max {cfg.max_steps} steps")
print(f"Robot: {cfg.robot} in {cfg.env_name}")
print(f"Stereo: L={cfg.left_camera}, R={cfg.right_camera}, baseline={cfg.stereo_baseline}m, full_3d={not cfg.stereo_use_table_z}")
print(f"Seed: {cfg.seed}")
print(f"Detection: OWLv2, queries={cfg.detection_queries}")

---
## 5. DSL Specification: Robot Motion Description Language (RMDL)

### Formal Syntax (JSON-based)

Each action primitive is a JSON object. The VLM outputs a list of these primitives.

```
move_to ::= { "type": "move_to", "target": "object" | "above_object", "height"?: float, "speed"?: float }
  → Code computes exact coordinates from stereo 3D detection. No manual offset arithmetic.

move_by ::= { "type": "move_by", "offset": [dx, dy, dz], "speed": float }
  → Relative move by [dx,dy,dz] meters. Use for simple adjustments with round numbers only.

descend ::= { "type": "descend" }
  → Descend vertically (maintain XY) until surface contact is detected (auto-stops).

rotate ::= { "type": "rotate", "rotation": [dr, dp, dy], "speed": float }
grasp ::= { "type": "grasp" }
release ::= { "type": "release" }
wait ::= { "type": "wait" }
```

### Semantics

| Action | Behavior | Target Resolution |
|--------|----------|-------------------|
| move_to (object) | Move EE to detected object position | Code: `target = object_3d` |
| move_to (above_object) | Move EE above detected object | Code: `target = object_3d + [0, 0, height]` |
| move_by | Move EE by relative offset | VLM: simple round numbers only |
| descend | Descend vertically until blocked | Code: `target_z = table_height` |
| grasp | Close gripper fully | — |
| release | Open gripper fully | — |
| rotate | Rotate EE by [dr,dp,dy] | — |
| wait | Do nothing | — |

### Key Design Principle

**Code computes coordinates, VLM decides strategy.**
- `move_to`: Code does the math (object_3d → exact EE target). VLM just says "go to object".
- `move_by`: VLM provides simple offsets for relative moves (e.g., lift 15cm → `[0,0,0.15]`).
- `descend`: Code handles vertical descent with plateau detection. VLM just says "descend".

In [ ]:
DSL_ACTION_TYPES = {"move_to", "move_by", "descend", "rotate", "grasp", "release", "wait"}

def validate_action(action: dict) -> bool:
    if not isinstance(action, dict): return False
    atype = action.get("type")
    if atype not in DSL_ACTION_TYPES: return False

    if atype == "move_by":
        offset = action.get("offset")
        if not isinstance(offset, list) or len(offset) != 3:
            return False
        if not all(isinstance(v, (int, float)) for v in offset):
            return False
        if "speed" in action:
            s = action["speed"]
            if not isinstance(s, (int, float)) or s < 0.1 or s > 1.0: return False

    elif atype == "move_to":
        target = action.get("target")
        if target not in ("object", "above_object"):
            return False
        if "height" in action:
            h = action["height"]
            if not isinstance(h, (int, float)) or h < 0 or h > 1.0: return False
        if "speed" in action:
            s = action["speed"]
            if not isinstance(s, (int, float)) or s < 0.1 or s > 1.0: return False

    elif atype == "descend":
        if "speed" in action:
            s = action["speed"]
            if not isinstance(s, (int, float)) or s < 0.1 or s > 1.0: return False

    elif atype == "rotate":
        rot = action.get("rotation")
        if not isinstance(rot, list) or len(rot) != 3:
            return False
        if any(not isinstance(v, (int, float)) or v < -1 or v > 1 for v in rot):
            return False
        if "speed" in action:
            s = action["speed"]
            if not isinstance(s, (int, float)) or s < 0.1 or s > 1.0: return False

    # grasp, release, wait — no required parameters
    return True

def validate_plan(plan: dict) -> bool:
    if not isinstance(plan, dict): return False
    if "actions" not in plan or not isinstance(plan["actions"], list): return False
    if len(plan["actions"]) == 0: return False
    if not all(validate_action(a) for a in plan["actions"]): return False
    return True

def validate_bbox(bbox) -> bool:
    """Check bbox is exactly 4 normalized values [x1,y1,x2,y2], or valid string tag."""
    if bbox is None: return True
    if isinstance(bbox, str): return bbox in ("no object found", "not necessary")
    if not isinstance(bbox, list): return False
    if len(bbox) != 4: return False
    if any(not isinstance(v, (int, float)) for v in bbox): return False
    if any(v < 0 or v > 1 for v in bbox): return False
    if bbox[2] <= bbox[0] or bbox[3] <= bbox[1]: return False
    return True

def validate_stereo_bbox(plan: dict) -> bool:
    """Validate that a detection result has both left_bbox and right_bbox."""
    if not isinstance(plan, dict): return False
    lb = plan.get("left_bbox")
    rb = plan.get("right_bbox")
    return validate_bbox(lb) and validate_bbox(rb)

# Smoke tests
assert validate_action({"type":"move_to","target":"object"})
assert validate_action({"type":"move_to","target":"above_object","height":0.05,"speed":0.5})
assert validate_action({"type":"move_by","offset":[0,0,0.15],"speed":0.5})
assert not validate_action({"type":"move","offset":[0,0,-0.1],"speed":0.3})  # "move" is REJECTED — must be "move_by"
assert validate_action({"type":"descend"})
assert not validate_action({"type":"move_to","target":"invalid"})
assert not validate_action({"type":"move_by","offset":[0,0]})

_sample = {"reasoning":"test","actions":[
    {"type":"move_to","target":"above_object","height":0.05},
    {"type":"descend"},
    {"type":"grasp"},
    {"type":"move_by","offset":[0,0,0.15],"speed":0.5}]}
assert validate_plan(_sample), "DSL validation failed"
assert validate_stereo_bbox({"left_bbox":[0.3,0.3,0.5,0.5],"right_bbox":[0.4,0.3,0.6,0.5]})
print("DSL definition OK")

---
## 6. VLM Interface (Qwen3-VL)

In [ ]:
class VLMPlanner:
    def __init__(self, model_id: str, use_4bit: bool = True):
        print(f"Loading Qwen3-VL: {model_id} ...")
        kwargs = {"torch_dtype": torch.float16, "device_map": "auto",
                  "trust_remote_code": True}
        if use_4bit and torch.cuda.is_available():
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
            kwargs["torch_dtype"] = torch.bfloat16
        self.model = Qwen3VLForConditionalGeneration.from_pretrained(
            model_id, **kwargs).eval()
        self.processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        self._call_count = 0
        print("  VLM loaded")

    # ─── Core call ──────────────────────────────────────────

    def _call(self, images: dict, prompt: str, max_new_tokens: int = 1024) -> str:
        content = []
        # Current stereo views
        for name in cfg.camera_names:
            if name in images:
                content.append({"type": "image", "image": images[name]})
        # Execution history snapshots (oldest first)
        history_keys = sorted([k for k in images.keys() if k.startswith("history_")])
        for key in history_keys:
            content.append({"type": "image", "image": images[key]})
        content.append({"type": "text", "text": prompt})
        conversation = [{"role": "user", "content": content}]
        inputs = self.processor.apply_chat_template(
            conversation, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt",
        ).to(self.model.device)
        inputs.pop("token_type_ids", None)
        with torch.no_grad():
            if SEED is not None:
                torch.manual_seed(SEED + self._call_count)
            self._call_count += 1
            ids = self.model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=True, temperature=0.3, top_p=0.9,
            )
        ids_trim = ids[0][inputs["input_ids"].shape[1]:]
        result = self.processor.decode(ids_trim, skip_special_tokens=True)
        print(f"--- VLM Call #{self._call_count} ---", flush=True)
        if vlm_log_path:
            with open(vlm_log_path, 'a') as f:
                f.write(f"--- VLM Call #{self._call_count} ---\n{result}\n\n")
        return result

    # ─── Phase 1: Stereo bbox detection ─────────────────────

    
    
    def _make_initial_prompt(self, task, steps_done, completed_actions, ee_pos, n_actions,
                            object_3d, gripper_state="open", needs_object=True):
        obj_str = ("unknown (no target detection)" if object_3d is None
                   else f"({object_3d[0]:.3f}, {object_3d[1]:.3f}, {object_3d[2]:.3f})")
        padded = list(completed_actions[-5:]) if completed_actions else []
        while len(padded) < 5: padded.append("---")
        completed_str = "; ".join(padded)
        placeholder_note = " (--- = no action yet)" if "---" in completed_str else ""
        cam_desc = build_images_desc(cfg.camera_names)
        return (
            f'[ROLE] Follow the rules below, use the specified action primitives, and output your plan in the required JSON format.\n'
            f'Your task: "{task}"\n'
            'Robot: Panda 7-DOF + parallel gripper. Controller: OSC_POSE (EE position control).\n'
            '\n'
            f'[STATUS] Step {steps_done}/{cfg.max_steps}. Task "{task}" is still NOT complete.\n'
            f'{len(completed_actions)} actions executed — still NOT complete.\n'
            + '[CURRENT STATE] (coordinates are approximate — cross-check with images)\n'
            f'  EE position: ({ee_pos[0]:.3f}, {ee_pos[1]:.3f}, {ee_pos[2]:.3f})\n'
            f'  Target 3D: {obj_str}  (move_to "target" is the keyword "object"/"above_object", NOT these coordinates)\n'
            f'  Gripper: {gripper_state}\n'
            f'  Recent actions: {completed_str}{placeholder_note}\n'
            '\n' + cam_desc + '\n'
            + _COORDS_DESC + '\n' + _ACTION_REFERENCE + '\n'
            + _build_availability_note(object_3d, needs_object) + '\n'
            + _STRATEGY + '\n'
            + f'[OUTPUT] You MUST output EXACTLY {n_actions} actions — fill every slot. Use wait for any slot whose\n'
            f'  real action you cannot confirm yet; it will be handled in the next re-check round.\n'
            + _SHARED_RULES + '\n'
            + _build_example(object_3d, n_actions))

    def _make_closedloop_prompt(self, task, steps_done, completed_actions, ee_pos, n_actions,
                                object_3d, gripper_state="unknown", history_desc=None, needs_object=True):
        obj_str = ("unknown (no target detection)" if object_3d is None
                   else f"({object_3d[0]:.3f}, {object_3d[1]:.3f}, {object_3d[2]:.3f})")
        padded = list(completed_actions[-5:]) if completed_actions else []
        while len(padded) < 5: padded.append("---")
        completed_str = "; ".join(padded)
        placeholder_note = " (--- = no action yet)" if "---" in completed_str else ""
        cam_desc = build_images_desc(cfg.camera_names)
        hist_block = ("\n" + history_desc) if history_desc else ""
        return (
            f'[ROLE] Follow the rules below, use the specified action primitives, and output your plan in the required JSON format.\n'
            'A previous sequence of actions was executed but the task is still not complete. You must continue executing.\n'
            f'Your task: "{task}"\n'
            'Robot: Panda 7-DOF + parallel gripper. Controller: OSC_POSE (EE position control).\n'
            '\n'
            f'[STATUS] Step {steps_done}/{cfg.max_steps}. Task "{task}" is still NOT complete.\n'
            f'{len(completed_actions)} actions executed — still NOT complete.\n'
            + '[CURRENT STATE] (coordinates are approximate — cross-check with images)\n'
            f'  EE position: ({ee_pos[0]:.3f}, {ee_pos[1]:.3f}, {ee_pos[2]:.3f})\n'
            f'  Target 3D: {obj_str}  (move_to "target" is the keyword "object"/"above_object", NOT these coordinates)\n'
            f'  Gripper: {gripper_state}\n'
            f'  Recent actions: {completed_str}{placeholder_note}\n'
            '\n' + cam_desc + hist_block + '\n'
            + _COORDS_DESC + '\n' + _ACTION_REFERENCE + '\n'
            + _build_availability_note(object_3d, needs_object) + '\n'
            + _STRATEGY + '\n'
            + f'[OUTPUT] You MUST output EXACTLY {n_actions} actions — fill every slot. Use wait for any slot whose\n'
            f'  real action you cannot confirm yet; it will be handled in the next re-check round.\n'
            + _SHARED_RULES + '\n'
            + _build_example(object_3d, n_actions))

    def analyze_task(self, left_img, right_img, task: str) -> tuple:
        """First VLM call: decide whether this task needs object detection, and name the target.
        Returns (needs_object_detection: bool, target_description: str | None).
        On any parse failure defaults to (True, None) so manipulation tasks never skip detection."""
        images = {cfg.left_camera: Image.fromarray(left_img),
                  cfg.right_camera: Image.fromarray(right_img)}
        prompt = (
            '[TASK ANALYSIS]\n'
            f'User task: "{task}"\n'
            'Look at the stereo images and the task. Decide whether completing this task requires '
            'detecting and locating a SPECIFIC physical object in the scene.\n'
            'YES (needs_object_detection=true): the task asks to pick up, grasp, push, touch, point at, '
            'or otherwise manipulate a named object (e.g. a cube, block, or cup).\n'
            'NO  (needs_object_detection=false): the task only needs the arm to move or rotate freely with '
            'NO specific object target (e.g. dance, wave, move around, rotate in place).\n'
            'Output JSON ONLY:\n'
            '{"reasoning":"...","needs_object_detection":true,"target_description":"red cube"}\n'
            '- target_description: the SINGLE object name to locate first when detection is needed (e.g. "red cube"). '
            'If the task involves several objects, name only the one needed FIRST — later targets are located in '
            'subsequent re-check rounds, one per round. Set to null when needs_object_detection is false.'
        )
        raw = self._call(images, prompt, max_new_tokens=256)
        parsed = self._extract_json(raw)
        if isinstance(parsed, dict) and "needs_object_detection" in parsed:
            needs = bool(parsed.get("needs_object_detection"))
            target = parsed.get("target_description")
        else:
            print("  [TASK ANALYSIS] parse failed, defaulting to needs=true", flush=True)
            needs, target = True, None
        if isinstance(target, str):
            target = target.strip()
            if not target or len(target) > 50:
                target = None
        else:
            target = None
        print(f"  [TASK ANALYSIS] needs_detection={needs}, target={target!r}", flush=True)
        if vlm_log_path:
            with open(vlm_log_path, 'a') as f:
                f.write(f"  [TASK ANALYSIS] needs={needs}, target={target!r}\n\n")
        return needs, target

    def regenerate_query(self, left_img, right_img, task, steps_done, completed_actions, ee_pos,
                         gripper_state="unknown", history_images=None, history_desc=None):
        """Per re-check round: given current execution state, output EXACTLY ONE OWLv2 query naming the
        single object to locate next. Multi-target tasks are handled one object per round (never a list).
        Returns the query string, or None on parse failure (caller keeps the previous round's query)."""
        images = {cfg.left_camera: Image.fromarray(left_img),
                  cfg.right_camera: Image.fromarray(right_img)}
        if history_images:
            for k, v in history_images.items():
                if isinstance(v, np.ndarray):
                    history_images[k] = Image.fromarray(v)
            images.update(history_images)
        padded = list(completed_actions[-5:]) if completed_actions else []
        while len(padded) < 5: padded.append("---")
        completed_str = "; ".join(padded)
        placeholder_note = " (--- = no action yet)" if "---" in completed_str else ""
        cam_desc = build_images_desc(cfg.camera_names)
        hist_block = ("\n" + history_desc) if history_desc else ""
        prompt = (
            '[QUERY RE-CHECK]\n'
            f'User task: "{task}"\n'
            'You are mid-execution in a closed loop that re-checks every few actions and re-plans. The task\n'
            'may take several re-check rounds to finish, and locating different objects is done ONE object\n'
            'per round — never more.\n'
            f'[STATUS] Step {steps_done}/{cfg.max_steps}. {len(completed_actions)} actions executed so far.\n'
            f'  EE position: ({ee_pos[0]:.3f}, {ee_pos[1]:.3f}, {ee_pos[2]:.3f})\n'
            f'  Gripper: {gripper_state}\n'
            f'  Recent actions: {completed_str}{placeholder_note}\n'
            '\n' + cam_desc + hist_block + '\n'
            '[DECIDE] Using the images (current views + history snapshots) and the completed work above,\n'
            'name the SINGLE physical object whose location is still needed to keep progressing toward the task.\n'
            'If the object you have been tracking is already secured (e.g. it has been grasped and is held off\n'
            'the table) or its location is already known and stable, SWITCH to the next object the task requires.\n'
            'Do not keep re-locating an object you already have. One object only — later rounds handle the rest.\n'
            'Output JSON ONLY:\n'
            '{"reasoning":"why this object is the right next target, given what is already done","query":"<the next object>"}\n'
            '- "query": ONE short noun phrase naming exactly ONE object. Name the actual next object for THIS\n'
            '  task from the images/task text — do NOT copy the placeholder or any example. Never a list, never\n'
            '  more than one object.'
        )
        raw = self._call(images, prompt, max_new_tokens=160)
        parsed = self._extract_json(raw)
        query = None
        if isinstance(parsed, dict):
            query = parsed.get("query") or parsed.get("target_description")
        if isinstance(query, str):
            query = query.strip().strip('"').strip("'")
            if not query or len(query) > 50:
                query = None
        else:
            query = None
        print(f"  [QUERY RE-CHECK] next target={query!r}", flush=True)
        if vlm_log_path:
            with open(vlm_log_path, 'a') as f:
                f.write(f"  [QUERY RE-CHECK] next target={query!r}\n\n")
        return query

    def plan_initial(self, left_img, right_img, task, steps_done, completed_actions, ee_pos,
                     n_actions=4, object_3d=None, gripper_state="open", needs_object=True):
        images = {cfg.left_camera: Image.fromarray(left_img),
                  cfg.right_camera: Image.fromarray(right_img)}
        prompt = self._make_initial_prompt(task, steps_done, completed_actions, ee_pos,
                                           n_actions, object_3d, gripper_state=gripper_state,
                                           needs_object=needs_object)
        return self._plan_with_retry(images, prompt, n_actions, indent="  ", closedloop=False)

    def plan_closedloop(self, left_img, right_img, task, steps_done, completed_actions, ee_pos,
                        n_actions, object_3d, gripper_state="unknown", history_images=None, history_desc=None,
                        needs_object=True):
        images = {cfg.left_camera: Image.fromarray(left_img),
                  cfg.right_camera: Image.fromarray(right_img)}
        # Convert any raw numpy history frames to PIL for consistent type
        if history_images:
            for k, v in history_images.items():
                if isinstance(v, np.ndarray):
                    history_images[k] = Image.fromarray(v)
            images.update(history_images)
        prompt = self._make_closedloop_prompt(
            task, steps_done, completed_actions, ee_pos, n_actions,
            object_3d, gripper_state=gripper_state, history_desc=history_desc,
            needs_object=needs_object)
        return self._plan_with_retry(images, prompt, n_actions, indent="    ", closedloop=True)

    @staticmethod
    def _action_hint(a):
        """Targeted guidance for the most common action-validation failures, shown back on retry."""
        if not isinstance(a, dict) or a.get("type") not in DSL_ACTION_TYPES:
            return f'Unknown/missing "type". Use one of: {", ".join(DSL_ACTION_TYPES)}.'
        t = a["type"]
        if t == "move_to" and a.get("target") not in ("object", "above_object"):
            return ('move_to "target" must be exactly "object" or "above_object" (a KEYWORD) — '
                    'NEVER coordinates or a tuple; the system computes exact XYZ from the detected object.')
        if t == "move_by" and not (isinstance(a.get("offset"), list) and len(a["offset"]) == 3):
            return 'move_by needs "offset":[dx,dy,dz] as three numbers (meters).'
        if t == "rotate" and not (isinstance(a.get("rotation"), list) and len(a["rotation"]) == 3):
            return 'rotate needs "rotation":[roll,pitch,yaw], each in [-1,1].'
        return ""

    def _plan_with_retry(self, images, prompt, n_actions, indent="  ", closedloop=False):
        """Call the VLM and retry on malformed/short plans (shared by plan_initial & plan_closedloop).
        Returns a validated plan with exactly n_actions actions, or None after 10 attempts.
        indent: console-print depth (initial=2sp, closedloop=4sp); closedloop enables the stronger
        retry message used when a response carries an 'error' field."""
        attempt = 0
        while True:
            if attempt >= 10:
                tag = "closed-loop" if closedloop else "initial"
                print(f"{indent}[PLAN] gave up after 10 attempts ({tag})", flush=True)
                return None
            raw = self._call(images, prompt, max_new_tokens=768)
            plan = self._extract_json(raw)
            if plan is not None and validate_plan(plan):
                if len(plan["actions"]) != n_actions:
                    err = f"expected {n_actions} actions, got {len(plan['actions'])}"
                    print(f"{indent}[PLAN] retry {attempt+1}: {err}", flush=True)
                    prompt += f'\n\nERROR: {err}. You MUST output exactly {n_actions} actions.'
                    attempt += 1
                    continue
                rsn = plan.get("reasoning", "")
                if rsn: print(f"{indent}[PLAN] {rsn}", flush=True)
                print(f"{indent}[PLAN] actions ({attempt+1}):", flush=True)
                for a in plan["actions"]: print(f"{indent}  {json.dumps(a)}", flush=True)
                return plan
            err = "unknown"
            hint = ""
            if plan is None: err = "json_parse_failed"
            elif not isinstance(plan.get("actions"), list): err = "no_actions"
            elif len(plan["actions"]) == 0 and "error" in plan:
                err = f"error_field: {plan['error'][:120]}"
            else:
                bad_idx = next((i for i, a in enumerate(plan["actions"]) if not validate_action(a)), None)
                if bad_idx is not None:
                    bad = plan["actions"][bad_idx]
                    err = f"invalid_action_{bad_idx}: {json.dumps(bad)}"
                    hint = VLMPlanner._action_hint(bad)
                else:
                    err = "empty_actions"
            print(f"{indent}[PLAN] retry {attempt+1}: {err}" + (f" | hint: {hint}" if hint else ""), flush=True)
            if closedloop and "error" in err:
                prompt += ('\n\nERROR: Your response contains an "error" field and empty actions. '
                           f'You MUST output EXACTLY {n_actions} valid actions. '
                           'NO error field. NO empty array.')
            else:
                prompt += (f"\n\nERROR: {err}. " +
                           (hint + " " if hint else "") +
                           f"Return valid JSON with exactly {n_actions} actions.")
            attempt += 1

    # ─── JSON extraction ────────────────────────────────────

    @staticmethod
    def _find_matching_brackets(text, open_ch, end_ch):
        """Find outermost matched brackets, respecting string literals."""
        start = text.find(open_ch)
        if start < 0:
            return None
        depth = 0
        in_string = False
        escape_next = False
        for i in range(start, len(text)):
            c = text[i]
            if escape_next:
                escape_next = False
                continue
            if c == '\\':
                escape_next = True
                continue
            if c == '"':
                in_string = not in_string
                continue
            if not in_string:
                if c == open_ch:
                    depth += 1
                elif c == end_ch:
                    depth -= 1
                    if depth == 0:
                        return text[start:i + 1]
        return None

    @staticmethod
    def _extract_json(text: str) -> Optional[dict]:
        text = text.strip()

        if text.startswith("{"):
            try:
                result = json.loads(text)
            except json.JSONDecodeError:
                result = None
            if result is not None:
                if isinstance(result.get("actions"), list) and len(result["actions"]) == 0:
                    _pat = re.compile(r'"actions"\s*:\s*(\[)', re.DOTALL)
                    for _m in _pat.finditer(text):
                        _start = _m.start(1)
                        _depth, _i = 1, _start + 1
                        while _depth > 0 and _i < len(text):
                            if text[_i] == '[': _depth += 1
                            elif text[_i] == ']': _depth -= 1
                            _i += 1
                        try:
                            _alt = json.loads(text[_start:_i])
                            if isinstance(_alt, list) and len(_alt) > 0:
                                result["actions"] = _alt
                                break
                        except (json.JSONDecodeError, TypeError):
                            continue
                return result

        if text.startswith("["):
            try:
                arr = json.loads(text)
                if isinstance(arr, list) and len(arr) > 0:
                    if all(isinstance(item, dict) and "type" in item for item in arr):
                        return {"actions": arr}
            except (json.JSONDecodeError, TypeError):
                pass

        obj_str = VLMPlanner._find_matching_brackets(text, '{', '}')
        if obj_str:
            try:
                return json.loads(obj_str)
            except json.JSONDecodeError:
                pass

        arr_str = VLMPlanner._find_matching_brackets(text, '[', ']')
        if arr_str:
            try:
                arr = json.loads(arr_str)
                if isinstance(arr, list) and len(arr) > 0:
                    if all(isinstance(item, dict) and "type" in item for item in arr):
                        return {"actions": arr}
            except (json.JSONDecodeError, TypeError):
                pass

        return None

    def unload(self):
        del self.model
        torch.cuda.empty_cache()
        print("  VLM unloaded")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Shared prompt constants (identical in initial + closed-loop prompts)
# ═══════════════════════════════════════════════════════════

_half_b = cfg.stereo_baseline / 2.0

_CAMERA_INFO = {
    "left_eye": (
        f'LEFT stereo eye:\n'
        f'   Front-side view from the robot\'s LEFT side (+{_half_b:.2f}m in Y).\n'
        f'   Image is in MuJoCo native orientation: table at BOTTOM, robot arm at TOP.\n'
        f'   LEFT = robot\'s LEFT (+Y), RIGHT = robot\'s RIGHT (−Y).\n'
        f'   Objects appear shifted RIGHT compared to the right eye view.\n'
    ),
    "right_eye": (
        f'RIGHT stereo eye:\n'
        f'   Front-side view from the robot\'s RIGHT side (−{_half_b:.2f}m in Y).\n'
        f'   Same orientation as left eye, offset horizontally.\n'
        f'   Objects appear shifted LEFT compared to the left eye view.\n'
    ),
}


def build_camera_rule(cameras):
    n = len(cameras)
    names = ", ".join(cameras)
    if n == 1:
        return f'CRITICAL: Analyze the camera view ({names}) carefully. Your reasoning MUST reference it.'
    return f'CRITICAL: Analyze BOTH stereo views — LEFT eye ({cameras[0]}) and RIGHT eye ({cameras[1]}). Your reasoning MUST reference both views.'


def build_images_desc(cameras):
    n = len(cameras)
    suffix = "s" if n > 1 else ""
    desc = f'[IMAGES]\n{n} stereo image{suffix}:\n'
    for i, cam in enumerate(cameras, 1):
        info = _CAMERA_INFO.get(cam, "")
        if info:
            desc += f'{i}. {info}'
    return desc


_COORDS_DESC = (
    '[COORDINATES]\n'
    'X = forward (+ = away from robot), Y = robot-left (+ = left), Z = up (table surface z = 0.80)\n'
    'Image coords (MuJoCo native): y=0 = BOTTOM (table surface), y=1 = TOP (robot arm area)\n'
)

_ACTION_REFERENCE = (
    '[ACTIONS — all are JSON objects with "type" and parameters]\n'
    '\n'
    'move_to {"type":"move_to", "target":"object"}\n'
    '  Move EE directly to the detected object\'s 3D position. Code computes exact coordinates.\n'
    '  CRITICAL: "target" is a KEYWORD ("object" or "above_object") — NEVER a coordinate or tuple.\n'
    'move_to {"type":"move_to", "target":"above_object", "height":0.05, "speed":0.5}\n'
    '  Move EE above the detected object by "height" meters. Default height = 0.05 (5cm).\n'
    '  ONLY use move_to when Target 3D is known (not "unknown").\n'
    '\n'
    'move_by {"type":"move_by", "offset":[dx,dy,dz], "speed":float}\n'
    '  Move EE by [dx,dy,dz] meters relative to current position.\n'
    '  Use for simple relative moves with round numbers (e.g., lift up → offset:[0,0,0.15]).\n'
    '  speed 0.1-1.0.\n'
    '\n'
    'descend {"type":"descend"}\n'
    '  Descend vertically (maintain current XY) until arm is blocked by surface contact.\n'
    '  Use for final vertical approach, e.g. before grasping or contacting a surface.\n'
    '\n'
    'rotate {"type":"rotate", "rotation":[dr,dp,dy], "speed":float}\n'
    '  rotation=[dr,dp,dy] normalized [-1,1], relative to current orientation.\n'
    '  speed 0.1-1.0.\n'
    '\n'
    'grasp {"type":"grasp"} — Close gripper fully. No parameters needed.\n'
    '\n'
    'release {"type":"release"} — Open gripper fully. No parameters needed.\n'
    '\n'
    'wait {"type":"wait"} — Do nothing for a fixed duration. No parameters needed.\n'
)


def _build_availability_note(object_3d, needs_object=True):
    """Generate action availability note based on detection state and task needs."""
    if object_3d is not None:
        return (
            '[ACTION AVAILABILITY]\n'
            'move_to is AVAILABLE - Target 3D position is known from stereo detection.\n'
            'For ANY approach to this position (reach the object, or hover above a container to place into it),\n'
            'use move_to (target "object" or "above_object"). The system drives the arm to the EXACT detected\n'
            'coordinates — you do not estimate distance. Use move_by ONLY for relative adjustments UNRELATED to\n'
            'the detected position (e.g. lift straight up after a grasp). NEVER guess a move_by offset toward the\n'
            'detected target — the estimate is wrong and the arm lands short.\n'
        )
    if needs_object:
        return (
            '[ACTION AVAILABILITY]\n'
            'move_to is NOT available - object detection was needed but no target was found.\n'
            'You MUST use move_by with offset for all position changes.\n'
        )
    return (
        '[ACTION AVAILABILITY]\n'
        'This task does NOT require a specific object - no detection was run.\n'
        'move_to is NOT available. Use move_by (offset), rotate, and wait to move the arm freely through space.\n'
    )


def _build_example(object_3d, n_actions):
    """Generate format example based on whether target is detected."""
    if object_3d is not None:
        acts = [
            '{"type":"move_to","target":"above_object","height":0.05}',
            '{"type":"descend"}',
            '{"type":"grasp"}',
            '{"type":"move_by","offset":[0,0,0.15],"speed":0.5}',
        ]
    else:
        acts = [
            '{"type":"move_by","offset":[0.1,0.0,0.05],"speed":0.5}',
            '{"type":"rotate","rotation":[0.3,0.0,0.2],"speed":0.4}',
            '{"type":"move_by","offset":[0.0,0.1,-0.05],"speed":0.5}',
            '{"type":"wait"}',
        ]
    while len(acts) < n_actions:
        acts.append('{"type":"wait"}')
    acts = acts[:n_actions]
    acts_str = ",\n  ".join(acts)
    return (
        '[EXAMPLE — format only, do not copy literally]\n'
        '{"reasoning":"Plan actions based on current state.","actions":[\n'
        f'  {acts_str}\n'
        ']}'
    )


_SHARED_RULES = (
    '[RULES]\n'
    '1. NEVER output "actions":[] or add an "error" field. Both are REJECTED immediately.\n'
    '2. Speed: 0.2-0.4 for precision (descend/grasp/rotate), 0.5-0.8 for gross movement (move_by).\n'
    '3. Vary action types — avoid consecutive repeats of the same type.\n'
    '4. In your "reasoning", analyze the IMAGES (current views + history snapshots) as your primary source. '
    'Coordinates are approximate and for reference only — trust what you see in the images.\n'
    '5. ' + build_camera_rule(cfg.camera_names) + '\n'
    '6. JSON ONLY. No text outside the JSON.\n'
)

_STRATEGY = (
    '[STRATEGY]\n'
    'Plan only what you can confidently execute THIS round. The task may need several re-check rounds to\n'
    'finish — you are NOT expected to complete it in a single plan, and later rounds will continue from\n'
    'where you stop. Break complex tasks into small steps.\n'
    '- If manipulating a specific object (pick up, push, touch): use move_to to approach, descend for the\n'
    '  final vertical approach, then grasp/release.\n'
    '- When Target 3D is KNOWN, ALWAYS use move_to (target "object" to reach it, or "above_object" to hover\n'
    '  above it) for the approach — the arm goes to the EXACT detected coordinates. NEVER use move_by to guess\n'
    '  a direction/distance toward a detected target; your offset estimate is wrong and the arm undershoots.\n'
    '  move_by is only for relative adjustments unrelated to a detected position (e.g. lift straight up).\n'
    '- To PLACE a held object INTO a container (bin, box, bowl): the container position is detected, so use\n'
    '  move_to above_object to move directly above the container, then descend into it, then release. Do NOT\n'
    '  nudge toward the container with move_by.\n'
    '- If the task needs free motion (move/rotate the arm, dance, wave) with no specific object: use\n'
    '  move_by (offset) and rotate. Do NOT approach or grasp an object unless the task requires it.\n'
    '- Plan ONLY the next step toward the object that is currently detected. If a later step needs a\n'
    '  DIFFERENT object that has not been located yet, do NOT guess where it is — fill those slots with\n'
    '  wait; the next re-check round will locate that object and continue.\n'
    '- Match your actions to what the task actually asks for.\n'
)

print("Prompt constants OK")

In [ ]:
vlm_log_path = None

_vlm_planner = None

def get_planner():
    global _vlm_planner
    if _vlm_planner is None:
        gc.collect()
        torch.cuda.empty_cache()
        _vlm_planner = VLMPlanner(cfg.vlm_model_id, cfg.use_4bit)
    return _vlm_planner

print("VLMPlanner class defined")

In [ ]:
# ── BboxDetector (OWLv2 for open-vocabulary object detection) ──

class BboxDetector:
    """Open-vocabulary object detector using OWLv2.
    
    Replaces Qwen3-VL's bbox detection phase. OWLv2 is natively
    supported in transformers (no trust_remote_code needed) and
    detects objects based on text queries.
    """

    def __init__(self, model_id: str = "google/owlv2-base-patch16-ensemble"):
        print(f"Loading OWLv2: {model_id} ...")
        self.processor = Owlv2Processor.from_pretrained(model_id)
        self.model = Owlv2ForObjectDetection.from_pretrained(model_id).eval()
        if torch.cuda.is_available():
            self.model = self.model.cuda()
        self._img_size = None
        print("  OWLv2 loaded")

    @torch.no_grad()
    def _infer(self, left_img, right_img, queries, threshold):
        """Run OWLv2 once over the stereo pair. Returns (result_left, result_right, (W, H))."""
        H, W = left_img.shape[:2]
        self._img_size = H
        # MuJoCo renders row 0 = bottom = table surface; PIL flips it to image top, which coincides
        # with OWLv2's y=0 = top. So bboxes need NO vertical flip (see git history for the derivation).
        left_pil = Image.fromarray(left_img)
        right_pil = Image.fromarray(right_img)
        inputs = self.processor(text=[queries, queries], images=[left_pil, right_pil],
                                return_tensors="pt").to(self.model.device)
        outputs = self.model(**inputs)
        target_sizes = torch.tensor([[H, W], [H, W]]).to(self.model.device)
        results = self.processor.post_process_grounded_object_detection(
            outputs, target_sizes=target_sizes, threshold=threshold)
        return results[0], results[1], (W, H)

    def detect_candidates(self, left_img, right_img, queries=None, threshold=0.05, k=4):
        """OWLv2 over the stereo pair; return up to top-k candidate bboxes per eye.

        Returns (left_cands, right_cands); each is a list of (bbox_norm, score), score-descending.
        bbox_norm is [x1,y1,x2,y2] in normalized MuJoCo-native coords [0,1] (y=0 = table surface).
        The live closed-loop path uses THIS (not detect()) so the controller can pick the L/R pair
        that is 3D-consistent, instead of trusting each eye's argmax independently."""
        if queries is None:
            queries = cfg.detection_queries
        rL, rR, (W, H) = self._infer(left_img, right_img, queries, threshold)
        return self._topk(rL, W, H, k), self._topk(rR, W, H, k)

    @torch.no_grad()
    def detect(self, left_img, right_img, queries=None, threshold=0.05):
        """Single best (argmax) bbox per eye. Kept for diagnostics (diagnostic_check); the live path
        uses detect_candidates() + stereo matching in the controller."""
        left_cands, right_cands = self.detect_candidates(left_img, right_img, queries, threshold)
        if not left_cands or not right_cands:
            reason_parts = []
            if not left_cands:
                reason_parts.append("left no detection")
            if not right_cands:
                reason_parts.append("right no detection")
            print(f"  [OWLv2] {', '.join(reason_parts)} (queries={queries}, threshold={threshold})", flush=True)
            return None
        left_bbox, score_left = left_cands[0]
        right_bbox, score_right = right_cands[0]
        print(f"  [OWLv2] L={left_bbox} (score={score_left:.3f})", flush=True)
        print(f"  [OWLv2] R={right_bbox} (score={score_right:.3f})", flush=True)
        reasoning = f"OWLv2 detection (q={queries}, scores={score_left:.2f}/{score_right:.2f})"
        return left_bbox, right_bbox, reasoning

    @staticmethod
    def _normalize_bbox(box, img_w, img_h):
        """Pixel [x1,y1,x2,y2] -> normalized MuJoCo-native [0,1]; None if degenerate."""
        bbox_norm = [float(box[0] / img_w), float(box[1] / img_h),
                     float(box[2] / img_w), float(box[3] / img_h)]
        if bbox_norm[2] <= bbox_norm[0] or bbox_norm[3] <= bbox_norm[1]:
            print(f"  [OWLv2] invalid bbox: {bbox_norm} (raw pixel box: {box})", flush=True)
            return None
        return bbox_norm

    @staticmethod
    def _topk(result, img_w, img_h, k):
        """Top-k scored valid bboxes as [(bbox_norm, score), ...], score descending."""
        scores = result["scores"]
        if len(scores) == 0:
            return []
        order = torch.argsort(scores, descending=True)[:k]
        out = []
        for idx in order.tolist():
            box = result["boxes"][idx].tolist()
            nb = BboxDetector._normalize_bbox(box, img_w, img_h)
            if nb is not None:
                out.append((nb, float(result["scores"][idx].item())))
        return out

    def unload(self):
        del self.model
        del self.processor
        torch.cuda.empty_cache()
        print("  OWLv2 unloaded")


_detector = None

def get_detector():
    """Singleton accessor for the OWLv2 bbox detector."""
    global _detector
    if _detector is None:
        gc.collect()
        torch.cuda.empty_cache()
        _detector = BboxDetector()
    return _detector

print("BboxDetector class defined")


---
## 7. Geometry Utilities

In [ ]:
# ── 7a. Rendering & Helper Utilities ──

@contextlib.contextmanager
def _silence_stderr():
    """Redirect stderr at both Python and OS FD level."""
    old_stderr = sys.stderr
    null_file = open(os.devnull, "w")
    sys.stderr = null_file
    null_fd = os.open(os.devnull, os.O_WRONLY)
    old_fd = os.dup(2)
    os.dup2(null_fd, 2)
    os.close(null_fd)
    try:
        yield
    finally:
        os.dup2(old_fd, 2)
        os.close(old_fd)
        sys.stderr = old_stderr
        null_file.close()


def render_rgb(env, camera="agentview"):
    """Render camera image in MuJoCo native orientation.

    Returns a contiguous uint8 array (H×W×3).
    MuJoCo OpenGL convention: origin at bottom-left.
      y=0 in image → bottom of view → near camera → table surface
      y=max in image → top of view → far from camera → robot arm area
    Both VLM and video display use this same raw orientation.
    """
    if camera is None:
        return None
    obs = env._get_observations(force_update=True)
    return np.ascontiguousarray(obs[f"{camera}_image"][..., :3])


print("Helpers OK")

In [ ]:
# ── 7b. Stereo Camera Injection ──

import shutil

# Convergence target: table surface center
_STEREO_TARGET = np.array([0.0, 0.0, 0.80])


def _lookat_quat(cam_pos, target, world_up=None):
    """Compute MuJoCo quaternion (w x y z) for camera at cam_pos looking at target."""
    if world_up is None:
        world_up = np.array([0.0, 0.0, 1.0])
    cam_pos = np.asarray(cam_pos, dtype=float)
    target = np.asarray(target, dtype=float)

    forward = target - cam_pos
    forward /= np.linalg.norm(forward)

    right = np.cross(world_up, forward)
    right /= np.linalg.norm(right)

    up = np.cross(forward, right)

    # MuJoCo cam_R columns: [right, -up, -forward]  (camera looks along -Z)
    # Negate up so det(R)=+1 (right-handed frame).
    R = np.column_stack([right, -up, -forward])

    # Rotation matrix → quaternion (w, x, y, z)
    tr = R[0, 0] + R[1, 1] + R[2, 2]
    if tr > 0:
        s = 0.5 / np.sqrt(tr + 1.0)
        w = 0.25 / s
        x = (R[2, 1] - R[1, 2]) * s
        y = (R[0, 2] - R[2, 0]) * s
        z = (R[1, 0] - R[0, 1]) * s
    elif R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:
        s = 2.0 * np.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2])
        w = (R[2, 1] - R[1, 2]) / s
        x = 0.25 * s
        y = (R[0, 1] + R[1, 0]) / s
        z = (R[0, 2] + R[2, 0]) / s
    elif R[1, 1] > R[2, 2]:
        s = 2.0 * np.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2])
        w = (R[0, 2] - R[2, 0]) / s
        x = (R[0, 1] + R[1, 0]) / s
        y = 0.25 * s
        z = (R[1, 2] + R[2, 1]) / s
    else:
        s = 2.0 * np.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1])
        w = (R[1, 0] - R[0, 1]) / s
        x = (R[0, 2] + R[2, 0]) / s
        y = (R[1, 2] + R[2, 1]) / s
        z = 0.25 * s

    return f"{w:.6f} {x:.6f} {y:.6f} {z:.6f}"


def _inject_stereo_cameras_xml():
    """Inject left_eye and right_eye cameras into robosuite's table_arena.xml.
    Cameras converge toward the table center. Original file is backed up and restored."""
    xml_path = os.path.join(
        os.path.dirname(suite.__file__),
        'models', 'assets', 'arenas', 'table_arena.xml')
    with open(xml_path) as f:
        original = f.read()

    backup = xml_path + '.bak'
    shutil.copy2(xml_path, backup)

    half_b = cfg.stereo_baseline / 2.0
    x, z = 0.5, 1.35
    pos_L = [x, half_b, z]
    pos_R = [x, -half_b, z]
    quat_L = _lookat_quat(pos_L, _STEREO_TARGET)
    quat_R = _lookat_quat(pos_R, _STEREO_TARGET)

    stereo_block = (
        f'\n    <!-- Binocular stereo cameras (auto-injected, converge to table center) -->\n'
        f'    <camera mode="fixed" name="left_eye"  pos="{x} {half_b:.4f} {z}" quat="{quat_L}"/>\n'
        f'    <camera mode="fixed" name="right_eye" pos="{x} {-half_b:.4f} {z}" quat="{quat_R}"/>\n'
    )
    modified = original.replace('</worldbody>', stereo_block + '</worldbody>')
    # Enable shadows in the scene lighting
    modified = modified.replace('castshadow="false"', 'castshadow="true"')
    with open(xml_path, 'w') as f:
        f.write(modified)
    return xml_path, backup


def _restore_arena_xml(xml_path, backup_path):
    """Restore the original arena XML after env creation."""
    if os.path.exists(backup_path):
        shutil.move(backup_path, xml_path)


print("Stereo injection OK")

In [ ]:
# ── 7c. Camera Parameters & Backprojection ──

def get_camera_params(env, camera="agentview", size=384):
    """Extract intrinsic matrix K, position, rotation, and table Z for a camera."""
    cam_id = env.sim.model.camera_name2id(camera)
    fov = float(env.sim.model.cam_fovy[cam_id])
    f = size / (2.0 * math.tan(math.radians(fov) / 2.0))
    K = np.array([[f, 0, size / 2], [0, f, size / 2], [0, 0, 1]])
    cam_pos = env.sim.model.cam_pos[cam_id].copy()
    cam_R = env.sim.model.cam_mat0[cam_id].reshape(3, 3).copy()
    table_z = float(env.model.mujoco_arena.table_offset[2])
    return K, cam_pos, cam_R, table_z


def get_stereo_params(env):
    """Get camera params for both stereo eyes plus table Z."""
    K_L, pos_L, R_L, tz = get_camera_params(env, camera=cfg.left_camera)
    K_R, pos_R, R_R, _   = get_camera_params(env, camera=cfg.right_camera)
    return (K_L, pos_L, R_L), (K_R, pos_R, R_R), tz


def backproject_ray(pixel_u, pixel_v, K, cam_pos, cam_R):
    """Back-project a pixel to a ray in world coordinates.

    Args:
        pixel_u, pixel_v: pixel coords in standard image frame (origin top-left, +u right, +v down)
        K: 3x3 intrinsic matrix
        cam_pos: camera position in world frame
        cam_R: camera-to-world rotation (cam_mat0 reshaped to 3x3)

    Returns:
        (origin, direction) tuple — ray in world coordinates
    """
    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]
    # Image is standard orientation (origin top-left, +u right, +v down).
    # Camera looks along -Z, so the ray through pixel (u,v) in camera frame is [(u-cx)/fx, (v-cy)/fy, -1].
    d = cam_R @ np.array([(pixel_u - cx) / fx, (pixel_v - cy) / fy, -1.0])
    d = d / np.linalg.norm(d)
    return cam_pos.copy(), d

print("Backprojection OK")

In [ ]:
# ── 7d. Stereo Triangulation ──

def triangulate_rays(origin_L, d_L, origin_R, d_R):
    """Find the closest point between two rays via least-squares.

    Solves: minimize |origin_L + t*d_L - origin_R - s*d_R|^2
    """
    W = origin_R - origin_L
    d1d1 = np.dot(d_L, d_L)
    d1d2 = np.dot(d_L, d_R)
    d2d2 = np.dot(d_R, d_R)
    A = np.array([[d1d1, -d1d2], [d1d2, -d2d2]])
    det = np.linalg.det(A)
    if abs(det) < 1e-10:
        return None  # Degenerate: parallel rays
    b = np.array([np.dot(W, d_L), np.dot(W, d_R)])
    ts = np.linalg.solve(A, b)
    pt_L = origin_L + ts[0] * d_L
    pt_R = origin_R + ts[1] * d_R
    gap = np.linalg.norm(pt_L - pt_R)
    if gap > 0.05:  # 5cm sanity threshold
        return None
    return (pt_L + pt_R) / 2.0


def stereo_bbox_to_3d(left_bbox_norm, right_bbox_norm, img_size, env):
    """Triangulate full 3D position from dual stereo bboxes.

    Both bboxes are in normalized MuJoCo-native coords [x1,y1,x2,y2] as output by VLM.
    No coordinate flipping needed — VLM sees the same raw orientation as MuJoCo.
    Returns complete (X, Y, Z) from ray-ray intersection.
    """
    if (left_bbox_norm is None or right_bbox_norm is None
            or not isinstance(left_bbox_norm, (list, tuple)) or len(left_bbox_norm) != 4
            or not isinstance(right_bbox_norm, (list, tuple)) or len(right_bbox_norm) != 4):
        return None

    (K_L, pos_L, R_L), (K_R, pos_R, R_R), tz = get_stereo_params(env)

    # Use bbox CENTER — both rays should point at the same physical point
    left_cx = ((left_bbox_norm[0] + left_bbox_norm[2]) / 2.0) * img_size
    left_cy = ((left_bbox_norm[1] + left_bbox_norm[3]) / 2.0) * img_size
    right_cx = ((right_bbox_norm[0] + right_bbox_norm[2]) / 2.0) * img_size
    right_cy = ((right_bbox_norm[1] + right_bbox_norm[3]) / 2.0) * img_size

    o_L, d_L = backproject_ray(left_cx, left_cy, K_L, pos_L, R_L)
    o_R, d_R = backproject_ray(right_cx, right_cy, K_R, pos_R, R_R)

    point_3d = triangulate_rays(o_L, d_L, o_R, d_R)
    if point_3d is None:
        return None

    # Optional: snap Z to known table height (legacy mode)
    if cfg.stereo_use_table_z:
        point_3d[2] = tz

    return point_3d

print("Triangulation OK")

In [ ]:
# ── Configurable objects + custom Lift environment ──
# cfg.objects = { "<built-in name or .xml/.obj path>": (x,y) | "random" | {"pos":(x,y),"color":...}, ... }
#   - EMPTY dict {} / None -> EMPTY TABLE (no objects at all; parameters are the single source of truth)
#   - the FIRST entry becomes the task object (env.cube / env.cube_body_id)
#   - color is set per-entry via "color" (cube defaults to red); any other dict keys (bin_size, ...) forward to the ctor
# See load_new_3Dmodel_experience.md for the external-mesh import workflow.

from collections import OrderedDict
from robosuite.environments.manipulation.lift import Lift
from robosuite.models.objects import BoxObject, MujocoXMLObject
from robosuite.models.arenas import TableArena
from robosuite.models.tasks import ManipulationTask
from robosuite.utils.placement_samplers import UniformRandomSampler
from robosuite.utils.mjcf_utils import xml_path_completion
import robosuite.models.objects as _robo_objects

# color presets (rgba) - "cube" defaults to "red"; any object sets its color via "color" in its dict
OBJECT_COLORS = {
    "red":    [1.0, 0.0, 0.0, 1.0],
    "blue":   [0.0, 0.0, 1.0, 1.0],
    "green":  [0.0, 0.6, 0.0, 1.0],
    "purple": [0.5, 0.0, 0.5, 1.0],
    "yellow": [1.0, 0.85, 0.0, 1.0],
    "white":  [1.0, 1.0, 1.0, 1.0],
    "black":  [0.05, 0.05, 0.05, 1.0],
    "silver": [0.80, 0.80, 0.83, 1.0],
    "gray":   [0.50, 0.50, 0.53, 1.0],
}

# built-in (non-cube) models: dict key -> robosuite object class (native color).
# Every key below is verified importable + instantiable in this robosuite version.
# "cube" is handled separately in _build_object (its color defaults to red; set via "color").
_BUILTIN_XML = {
    # --- food / packaged (mesh assets; recommended pickable objects) ---
    "can":               "CanObject",
    "lemon":             "LemonObject",
    "bottle":            "BottleObject",
    "milk":              "MilkObject",
    "cereal":            "CerealObject",
    "bread":             "BreadObject",
    # --- primitives (parametric; reliable table placement) ---
    "ball":              "BallObject",
    "cylinder":          "CylinderObject",
    "hollow_cylinder":   "HollowCylinderObject",
    "capsule":           "CapsuleObject",
    "cone":              "ConeObject",
    # --- tools / composite (from other task envs; larger footprint, may place oddly) ---
    "hammer":            "HammerObject",
    "pot_with_handles":  "PotWithHandlesObject",
    "bin":               "Bin",
    "hinged_box":        "HingedBoxObject",
    "ratcheting_wrench": "RatchetingWrenchObject",
    "round_nut":         "RoundNutObject",
    "square_nut":        "SquareNutObject",
    "plate_with_hole":   "PlateWithHoleObject",
    "door":              "DoorObject",
}


def _xml_object_from_obj(obj_path, name):
    """Wrap an external .obj mesh in a MuJoCo XML (mesh + material + required sites)
    and return a MujocoXMLObject. Mesh file is resolved relative to the wrapper XML."""
    obj_file = os.path.basename(obj_path)
    xml_path = obj_path + ".wrap.xml"
    xml = (
        '<mujoco model="custom_obj">\n'
        '  <asset>\n'
        f'    <mesh file="{obj_file}" name="mesh" scale="1.0 1.0 1.0"/>\n'
        f'    <material name="mat" rgba="0.8 0.4 0.2 1" reflectance="0.3"/>\n'
        '  </asset>\n'
        '  <worldbody>\n'
        '    <body name="object">\n'
        '      <geom pos="0 0 0" mesh="mesh" type="mesh" material="mat"\n'
        '            density="500" friction="0.95 0.3 0.1"/>\n'
        '    </body>\n'
        '    <site rgba="0 0 0 0" size="0.005" pos="0 0 -0.01" name="bottom_site"/>\n'
        '    <site rgba="0 0 0 0" size="0.005" pos="0 0 0.05" name="top_site"/>\n'
        '    <site rgba="0 0 0 0" size="0.005" pos="0.03 0 0.025" name="horizontal_radius_site"/>\n'
        '  </worldbody>\n'
        '</mujoco>\n'
    )
    with open(xml_path, "w") as _f:
        _f.write(xml)
    return MujocoXMLObject(xml_path, name=name,
                           joints=[dict(type="free", damping="0.0005")],
                           obj_type="all", duplicate_collision_geoms=True)


def _parse_obj_value(val):
    """Normalize a cfg.objects value into (pos, color, kwargs, obj_range).

    Accepted forms:
      (x, y) | "random"                  -> ((x,y) or "random", None, {}, None)
      {"pos": (x,y)|"random", "color": "red",
       "range": [(xmin,xmax),(ymin,ymax)],   # optional per-object random box (overrides cfg ranges)
       <object-kwarg>: value, ...}       -> (pos, color, {ctor kwargs}, range)
    "range" is NOT forwarded to the constructor; it fixes the random spawn box for this one
    object (used only when pos=="random"). Extra dict keys (bin_size, wall_thickness, size,
    ...) ARE forwarded to the object constructor."""
    if isinstance(val, dict):
        pos = val.get("pos")
        color = val.get("color")
        obj_range = val.get("range")
        kwargs = {k: v for k, v in val.items() if k not in ("pos", "color", "range")}
        return pos, color, kwargs, obj_range
    return val, None, {}, None


def _recolor_object(obj, rgba):
    """Override a mesh object's baked material/texture with a solid rgba."""
    wb = getattr(obj, "worldbody", None)
    if wb is None:
        return
    for _g in wb.findall(".//geom"):
        _g.attrib.pop("material", None)
        _g.set("rgba", " ".join(str(c) for c in rgba))


# composite classes whose look defaults to a baked texture; a named "color" must flip
# use_texture=False so the rgba actually shows instead of the texture.
_TEXTURED_COMPOSITE = {"bin", "pot_with_handles", "hammer", "hinged_box"}


def _build_object(spec, name, obj_color=None, obj_kwargs=None):
    """Build a MujocoObject from a spec.
    spec: 'cube' | built-in name (can/lemon/bin/...) | path to an existing .xml or .obj file.
    obj_color: optional color preset name (rgba); 'cube' defaults to red when unset.
    obj_kwargs: extra constructor kwargs (e.g. bin_size, wall_thickness, size) from the value dict."""
    obj_kwargs = dict(obj_kwargs or {})
    if str(spec).strip().lower() == "cube":
        rgba = OBJECT_COLORS.get(obj_color, OBJECT_COLORS["red"]) if obj_color else OBJECT_COLORS["red"]
        return BoxObject(name=name, size_min=[0.020, 0.020, 0.020], size_max=[0.022, 0.022, 0.022],
                         rgba=rgba)
    key = str(spec).strip().lower()
    rgba = OBJECT_COLORS.get(obj_color) if obj_color else None
    if key in _BUILTIN_XML:
        cls = getattr(_robo_objects, _BUILTIN_XML[key])
        ctor_kw = dict(obj_kwargs)
        if rgba is not None:
            ctor_kw.setdefault("rgba", rgba)
            if key in _TEXTURED_COMPOSITE:
                ctor_kw.setdefault("use_texture", False)
        try:
            return cls(name=name, **ctor_kw)
        except TypeError:
            # ctor rejects a kwarg (e.g. use_texture on a non-composite, or an unknown size param)
            # -> build bare, then apply color via geom rgba override if requested.
            obj = cls(name=name)
            if rgba:
                _recolor_object(obj, rgba)
            return obj
    path = str(spec)
    if os.path.exists(path) and path.lower().endswith(".xml"):
        obj = MujocoXMLObject(path, name=name,
                              joints=[dict(type="free", damping="0.0005")],
                              obj_type="all", duplicate_collision_geoms=True)
        if rgba:
            _recolor_object(obj, rgba)
        return obj
    if os.path.exists(path) and path.lower().endswith(".obj"):
        obj = _xml_object_from_obj(path, name)
        if rgba:
            _recolor_object(obj, rgba)
        return obj
    raise ValueError(
        f"unknown object spec {spec!r}: use 'cube', a built-in name {list(_BUILTIN_XML)}, "
        "or an existing .xml/.obj path")


class LiftCustom(Lift):
    """Lift task whose object(s) are configured ENTIRELY by a dict.

    objects_spec: {spec: (x,y) | "random" | {"pos":(x,y),"color":...,<kwarg>...}}; first entry is
                  the task object (self.cube). EMPTY dict -> EMPTY TABLE (self.cube=None).
                  Each value's "color" sets the object color (cube defaults to red); any other keys
                  (bin_size, wall_thickness, size, ...) are forwarded to that object's constructor.

    Empty-table guards: _setup_references / reward / _check_success short-circuit when
    self.cube is None so an object-free scene resets, renders, and steps without crashing."""

    def __init__(self, *args, objects_spec=None, **kwargs):
        # No secret default: empty/None -> empty table. Parameters are the single source of truth.
        self._objects_spec = OrderedDict(objects_spec) if objects_spec else OrderedDict()
        super().__init__(*args, **kwargs)

    def _load_model(self):
        # Skip Lift._load_model (hardcodes the red cube); rebuild the scene from cfg.objects.
        super(Lift, self)._load_model()
        xpos = self.robots[0].robot_model.base_xpos_offset["table"](self.table_full_size[0])
        self.robots[0].robot_model.set_base_xpos(xpos)
        mujoco_arena = TableArena(table_full_size=self.table_full_size,
                                  table_friction=self.table_friction,
                                  table_offset=self.table_offset)
        mujoco_arena.set_origin([0, 0, 0])

        self._obj_specs = []      # list of (name, spec, pos, obj_range)
        objs = []
        # Bin tracking: if a "bin" entry is present, record its configured size + wall thickness
        # so the controller can compute the bin interior (world coords) for the place-in-bin
        # success test. Defaults match robosuite Bin.__init__.
        self.bin_obj = None
        self.bin_size = (0.3, 0.3, 0.15)
        self.bin_wall_thickness = 0.01
        for i, (spec, val) in enumerate(self._objects_spec.items()):
            pos, color, obj_kwargs, obj_range = _parse_obj_value(val)
            name = "cube" if i == 0 else f"obj{i}"
            obj = _build_object(spec, name, obj_color=color, obj_kwargs=obj_kwargs)
            self._obj_specs.append((name, spec, pos, obj_range))
            objs.append(obj)
            if str(spec).strip().lower() == "bin":
                self.bin_obj = obj
                if "bin_size" in obj_kwargs:
                    self.bin_size = tuple(obj_kwargs["bin_size"])
                if "wall_thickness" in obj_kwargs:
                    self.bin_wall_thickness = float(obj_kwargs["wall_thickness"])
        self.cube = objs[0] if objs else None   # task object = first dict entry; None if empty
        self._all_objects = objs

        # Initial sample spans the full cfg ranges so N objects can obtain a valid
        # (non-overlapping) placement; the exact per-object position is then set
        # in _reset_internal from cfg.objects. Empty objs -> sampler returns {}.
        self.placement_initializer = UniformRandomSampler(
            name="ObjectSampler", mujoco_objects=objs,
            x_range=list(cfg.object_x_range), y_range=list(cfg.object_y_range),
            rotation=None,
            ensure_object_boundary_in_range=False, ensure_valid_placement=True,
            reference_pos=self.table_offset, z_offset=0.01, rng=self.rng)

        self.model = ManipulationTask(
            mujoco_arena=mujoco_arena,
            mujoco_robots=[r.robot_model for r in self.robots],
            mujoco_objects=objs,
        )

    def _setup_references(self):
        # Skip Lift._setup_references (it dereferences self.cube unconditionally); call
        # RobotEnv's version, then set cube_body_id only when a task object exists.
        super(Lift, self)._setup_references()
        self.cube_body_id = (self.sim.model.body_name2id(self.cube.root_body)
                             if self.cube is not None else None)
        # Bin body id (for the pick_and_place success test); None when the scene has no bin.
        self.bin_body_id = (self.sim.model.body_name2id(self.bin_obj.root_body)
                            if getattr(self, "bin_obj", None) is not None else None)

    def _setup_observables(self):
        # Empty table: skip Lift's cube sensors (cube_pos/cube_quat need a cube body id).
        if self.cube is None:
            return super(Lift, self)._setup_observables()
        return super()._setup_observables()

    def reward(self, action=None):
        # Empty table -> no task -> zero reward (never dereference self.cube).
        if self.cube is None:
            return 0.0
        return super().reward(action)

    def _check_success(self):
        if self.cube is None:
            return False
        return super()._check_success()

    def _obj_local_bottom_z(self, obj):
        """Lowest point of obj geometry in its body frame (z < 0 => below the body origin).
        Cube origins sit at the center; can/bottle/milk origins sit high above the base, so a
        uniform body z would clip tall objects. Reads every geom in the object body subtree
        (box/sphere/cylinder/capsule/ellipsoid/mesh) and returns the lowest extent, giving the
        true base for resting the object on the table regardless of origin convention or height."""
        bid = self.sim.model.body_name2id(obj.root_body)
        lowest = float("inf")
        for gi in range(self.sim.model.ngeom):
            gb = int(self.sim.model.geom_bodyid[gi])
            if int(self.sim.model.body_rootid[gb]) != bid:
                continue
            gtype = int(self.sim.model.geom_type[gi])
            gz = float(self.sim.model.geom_pos[gi][2])
            sz = self.sim.model.geom_size[gi]
            if gtype == 6:                       # box
                cand = gz - sz[2]
            elif gtype == 2:                     # sphere
                cand = gz - sz[0]
            elif gtype in (5, 3, 4):             # cylinder / capsule / ellipsoid
                cand = gz - sz[1]
            elif gtype == 7:                     # mesh -> min vertex z, offset by geom local z
                mid = int(self.sim.model.geom_dataid[gi])
                adr = int(self.sim.model.mesh_vertadr[mid])
                num = int(self.sim.model.mesh_vertnum[mid])
                verts = self.sim.model.mesh_vert[adr:adr + num]
                cand = gz + float(verts[:, 2].min())
            else:
                continue
            lowest = min(lowest, cand)
        return lowest if lowest != float("inf") else 0.0

    def _reset_internal(self):
        super()._reset_internal()      # base reset + initial placement (no-op when no objects)
        # Override placement per spec: fixed (x, y) or "random" within cfg ranges.
        for (name, spec, pos, obj_range), obj in zip(self._obj_specs, self._all_objects):
            if pos == "random":
                # Per-object "range" overrides the global cfg ranges so a single object's
                # random box can be restricted (e.g. keep the cube off the bin footprint).
                if obj_range is not None:
                    xr, yr = obj_range
                    x = self.rng.uniform(xr[0], xr[1])
                    y = self.rng.uniform(yr[0], yr[1])
                else:
                    x = self.rng.uniform(cfg.object_x_range[0], cfg.object_x_range[1])
                    y = self.rng.uniform(cfg.object_y_range[0], cfg.object_y_range[1])
            elif isinstance(pos, (tuple, list)) and len(pos) == 2:
                x, y = float(pos[0]), float(pos[1])
            else:
                continue
            # Rest the true BASE of each object on the table. Objects differ in height and in where
            # the body origin sits (cube origin = center; can/bottle/milk origins sit well above the
            # base), so a uniform z clips tall objects into the table and MuJoCo pops them out on
            # reset. _obj_local_bottom_z returns the lowest point of the geometry in body frame;
            # placing body_z at table_surface - local_bottom lands that lowest point on the table.
            obj_pos = np.array([x, y, self.table_offset[2] - self._obj_local_bottom_z(obj)])
            self.sim.data.set_joint_qpos(
                obj.joints[0], np.concatenate([obj_pos, np.array([0.0, 0.0, 0.0, 1.0])]))
        self.sim.forward()

print("LiftCustom + object factory OK")


In [ ]:
# ── 7e. Environment Creation ──

def make_env(cfg, seed=None, do_reset=True):
    """Create env with stereo cameras injected into the arena XML.
    The original XML is backed up and restored after model compilation."""
    cameras = list(dict.fromkeys(cfg.camera_names))
    xml_path, backup_path = _inject_stereo_cameras_xml()
    try:
        with _silence_stderr():
            cc = load_composite_controller_config(controller="BASIC")
            with open(os.devnull, "w") as _dn:
                with contextlib.redirect_stdout(_dn):
                    env = suite.make(
                        "LiftCustom", robots=cfg.robot, controller_configs=cc,
                        has_renderer=False, has_offscreen_renderer=True, renderer="mujoco",
                        camera_names=cameras,
                        camera_heights=cfg.camera_size, camera_widths=cfg.camera_size,
                        camera_depths=False,
                        use_camera_obs=True, reward_shaping=True, control_freq=20,
                        seed=seed if seed is not None else cfg.seed,
                        objects_spec=cfg.objects,
                    )
            if do_reset:
                env.reset()   # LiftCustom._reset_internal places each object per cfg.objects
    finally:
        _restore_arena_xml(xml_path, backup_path)
    # Hide Robosuite's gripper grip-axis "guide line": a long, thin, NON-physical SITE at the
    # end-effector (gripper0_right_grip_site_cylinder, type=5, half-length ~10 m, rgba alpha<0).
    # It is a visualization aid, not a scene object, yet it casts a stray vertical line-segment
    # shadow on the table/objects. Zero its alpha so neither the line nor its shadow renders;
    # every OTHER shadow is untouched (only this site's geometry is suppressed).
    try:
        _sid = env.sim.model.site_name2id("gripper0_right_grip_site_cylinder")
        env.sim.model.site_rgba[_sid] = [0.0, 1.0, 0.0, 0.0]
    except (ValueError, AttributeError):
        pass
    return env

print("make_env OK")

In [ ]:
# ── 7f. Stereo Geometry Verification ──

with _silence_stderr():
    test_env = make_env(cfg)

# Check camera positions
for cam_name in cfg.camera_names:
    cam_id = test_env.sim.model.camera_name2id(cam_name)
    pos = test_env.sim.model.cam_pos[cam_id].copy()
    print(f"  {cam_name}: ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")

# Both cameras must render different images (stereo disparity)
img_L = render_rgb(test_env, camera=cfg.left_camera)
img_R = render_rgb(test_env, camera=cfg.right_camera)
diff = np.sum(np.abs(img_L.astype(int) - img_R.astype(int)) > 5)
total = 384 * 384 * 3
print(f"  Pixel diff > 5: {diff}/{total} ({100*diff/total:.1f}%)")
assert diff > 0, "WARNING: left and right cameras produce identical images!"

# Synthetic perfect-bbox triangulation test
cube_pos = test_env.sim.data.body_xpos[test_env.cube_body_id].copy()
(K_L, pos_L, R_L), (K_R, pos_R, R_R), tz = get_stereo_params(test_env)


def _project(p3d, cam_pos, cam_R, K):
    """Project 3D point to MuJoCo native pixel coords (no flip)."""
    pc = cam_R.T @ (p3d - cam_pos)
    fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]
    u = cx - fx * pc[0] / pc[2]   # standard image coords (top-left origin, +v down)
    v = cy - fy * pc[1] / pc[2]
    return u, v


u_L, v_L = _project(cube_pos, pos_L, R_L, K_L)
u_R, v_R = _project(cube_pos, pos_R, R_R, K_R)

# Convert pixel centers to normalized bbox (small square around the point)
half_px = 20 / cfg.camera_size  # 20px half-size
synth_left_bbox = [u_L/384 - half_px, v_L/384 - half_px,
                   u_L/384 + half_px, v_L/384 + half_px]
synth_right_bbox = [u_R/384 - half_px, v_R/384 - half_px,
                    u_R/384 + half_px, v_R/384 + half_px]

result_3d = stereo_bbox_to_3d(synth_left_bbox, synth_right_bbox, cfg.camera_size, test_env)
assert result_3d is not None, "Triangulation returned None!"
xy_err = np.linalg.norm(result_3d[:2] - cube_pos[:2])
z_err = abs(result_3d[2] - cube_pos[2])
xyz_err = np.linalg.norm(result_3d - cube_pos)
print(f"  Cube true: ({cube_pos[0]:.4f}, {cube_pos[1]:.4f}, {cube_pos[2]:.4f})")
print(f"  Stereo 3D: ({result_3d[0]:.4f}, {result_3d[1]:.4f}, {result_3d[2]:.4f})")
print(f"  XY error:  {xy_err*100:.2f} cm")
print(f"  Z error:   {z_err*100:.2f} cm")
print(f"  3D error:  {xyz_err*100:.2f} cm")
assert xy_err < 0.01, f"Stereo XY error too large: {xy_err*100:.1f}cm"
assert z_err < 0.01, f"Stereo Z error too large: {z_err*100:.1f}cm"

test_env.close()
print("Stereo geometry OK")

---
## 8. VLM Controller (Closed-Loop)

In [ ]:
from dataclasses import dataclass
@dataclass
class TrialResult:
    success: bool
    steps: int
    error: Optional[str] = None
    plan: Optional[dict] = None

In [ ]:
class VideoCapture:
    def __init__(self, fps=8):
        self.frames = []
        self.fps = fps

    def _draw_bbox(self, img_np, bbox_norm, color, label=""):
        """Draw a normalized bbox rectangle on a numpy image (MuJoCo native coords)."""
        from PIL import Image, ImageDraw, ImageFont
        h, w = img_np.shape[:2]
        x1 = int(bbox_norm[0] * w)
        y1 = int(bbox_norm[1] * h)
        x2 = int(bbox_norm[2] * w)
        y2 = int(bbox_norm[3] * h)
        pil = Image.fromarray(img_np)
        draw = ImageDraw.Draw(pil)
        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
        if label:
            try:
                font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 11)
            except Exception:
                font = ImageFont.load_default()
            draw.text((x1 + 2, max(0, y1 - 14)), label, font=font, fill=color)
        return np.array(pil)

    def snap(self, left_img, right_img, phase="", plan=None, t3d=None, msg="",
             step=0, pause=False, bbox=None):
        import cv2
        from PIL import Image, ImageDraw, ImageFont
        left_img = np.ascontiguousarray(left_img)
        right_img = np.ascontiguousarray(right_img)

        # Draw bbox rectangles if provided: bbox = {"left": [x1,y1,x2,y2], "right": [x1,y1,x2,y2]}
        if bbox and isinstance(bbox, dict):
            if "left" in bbox and isinstance(bbox["left"], list):
                left_img = self._draw_bbox(left_img, bbox["left"], (0, 255, 0), "target")
            if "right" in bbox and isinstance(bbox["right"], list):
                right_img = self._draw_bbox(right_img, bbox["right"], (0, 255, 0), "target")

        bar_h = 20

        def _add_label(img_np, label):
            hh, ww = img_np.shape[:2]
            frame = np.zeros((hh + bar_h, ww, 3), dtype=np.uint8)
            frame[:hh] = img_np
            frame[hh:][:] = (40, 40, 50)
            pil_f = Image.fromarray(frame)
            try:
                font_b = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 13)
            except Exception:
                font_b = ImageFont.load_default()
            ImageDraw.Draw(pil_f).text((6, hh + 2), label, font=font_b, fill=(200, 230, 255))
            return np.array(pil_f)

        # Wrap each eye (image + its label bar) in a thin colored border so the two
        # views are visually separated instead of flush. Left=blue, Right=amber.
        eye_border = 4
        labeled = [
            cv2.copyMakeBorder(_add_label(left_img, "Left Eye"),
                               eye_border, eye_border, eye_border, eye_border,
                               cv2.BORDER_CONSTANT, value=(60, 120, 170)),
            cv2.copyMakeBorder(_add_label(right_img, "Right Eye"),
                               eye_border, eye_border, eye_border, eye_border,
                               cv2.BORDER_CONSTANT, value=(170, 120, 60)),
        ]
        disp = np.concatenate(labeled, axis=1)
        h, w = disp.shape[:2]
        info_h = 260
        composite = np.zeros((h + info_h, w, 3), dtype=np.uint8)
        composite[:h] = disp
        composite[h:][:] = (25, 25, 35)

        pil_img = Image.fromarray(composite)
        draw = ImageDraw.Draw(pil_img)
        try:
            font_m = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 14)
            font_s = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 11)
        except Exception:
            font_m = font_s = ImageFont.load_default()

        def _wrap(text, font, max_w):
            words = text.split()
            lines_o, cur = [], ""
            for wd in words:
                test = cur + " " + wd if cur else wd
                if draw.textbbox((0, 0), test, font=font)[2] <= max_w:
                    cur = test
                else:
                    if cur: lines_o.append(cur)
                    cur = wd
            if cur: lines_o.append(cur)
            return lines_o

        margin, max_text_w = 10, w - 2 * 10
        y = h + 8
        draw.text((margin, y), f"Step {step}  {phase}", font=font_m, fill=(255, 200, 100))
        y += 22
        if plan:
            for rl in _wrap(plan.get("reasoning", ""), font_s, max_text_w):
                draw.text((margin, y), rl, font=font_s, fill=(180, 220, 255)); y += 16
            for aidx, act in enumerate(plan.get("actions", [])):
                for al in _wrap(f"  {aidx+1}. {json.dumps(act, ensure_ascii=False)}", font_s, max_text_w):
                    draw.text((margin, y), al, font=font_s, fill=(150, 200, 150)); y += 16
                    if y > h + info_h - 16: break
                if y > h + info_h - 16:
                    draw.text((margin, y), "  ...", font=font_s, fill=(100, 100, 100)); break
        if t3d is not None:
            draw.text((margin, y), f"3D: ({t3d[0]:.3f}, {t3d[1]:.3f}, {t3d[2]:.3f})",
                      font=font_s, fill=(150, 200, 255)); y += 16
        if msg:
            for ml in _wrap(msg, font_s, max_text_w):
                draw.text((margin, y), ml, font=font_s, fill=(200, 200, 200)); y += 16
        self.frames.append(np.array(pil_img))
        if pause:
            for _ in range(12): self.frames.append(np.array(pil_img))  # 1.5s at 8fps

    def save(self, path="output.mp4"):
        n_frames = len(self.frames)
        print(f"  [SAVE] {n_frames} frames, {self.fps} fps", flush=True)
        if not self.frames:
            return path

        # Primary: imageio + imageio-ffmpeg. The bundled ffmpeg ships libx264, giving
        # clean H264 output and avoiding OpenCV's "Encoder not found" stderr spam that
        # OpenCV's own libavcodec (no libx264) prints for avc1/H264/X264 on Kaggle.
        try:
            import imageio
            writer = imageio.get_writer(path, fps=self.fps, codec="libx264",
                                        quality=9, macro_block_size=2)
            for f in self.frames:
                writer.append_data(f)
            writer.close()
            size = os.path.getsize(path)
            if size > 1024:
                print(f"  [SAVE] imageio/libx264: {size/1024:.0f} KB", flush=True)
                return path
        except Exception as e:
            print(f"  [SAVE] imageio libx264 failed ({e}); trying cv2 mp4v", flush=True)

        # Fallback: cv2 with mp4v (MPEG-4 Part 2). Goes straight to mp4v to avoid the
        # avc1/H264/X264 attempts that spam stderr on Kaggle's OpenCV build.
        import cv2
        h, w = self.frames[0].shape[:2]
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        out = cv2.VideoWriter(path, fourcc, self.fps, (w, h))
        ok = out.isOpened()
        if ok:
            for f in self.frames:
                out.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
        out.release()
        if ok and os.path.exists(path) and os.path.getsize(path) > 1024:
            print(f"  [SAVE] cv2/mp4v: {os.path.getsize(path)/1024:.0f} KB", flush=True)
            return path

        print("  [SAVE] All encoders failed!", flush=True)
        return path


In [ ]:
class VLMController:
    def __init__(self, env, planner: VLMPlanner, on_frame=None):
        self.env = env
        self.planner = planner
        eef = env.robots[0].eef_site_id
        self._ee_site = eef[list(eef.keys())[0]] if isinstance(eef, dict) else eef
        self._tz = 0.800      # table-surface z (success floor = _tz+0.05)
        self._grasp_z = 0.826  # descend target: pad-center aligns with cube center
        self.on_frame = on_frame
        self._step = 0
        self._action_count = 0
        self._history_frames = []       # list of ({camera: frame}, desc), oldest first, max 2
        self._gripper_closed = False     # track gripper state across actions
        self._target_cache = {}   # query -> 3D position; persists across re-plan rounds WITHIN a
                                  # trial (reset per trial in run_closedloop) so a failed re-detect
                                  # falls back to THIS query's last-known pose, never another
                                  # object's stale coordinates
        self._object_held = False  # latches True once a grasp + lift is observed (cube clearly off the
                                   # table with the gripper closed); stays True through an intentional
                                   # descend-to-place so the planner is never told the grasp failed
        self._actions = []          # recorded per-env.step actions (for offline replay)
        self._frames = []           # recorded frame-event metadata (for offline replay)

    def _ee_pos(self):
        return self.env.sim.data.site_xpos[self._ee_site].copy()

    def _osc_action(self, dxyz, grip, rotation=None):
        if rotation is None: rotation = [0, 0, 0]
        return np.concatenate([np.clip(dxyz / 0.05, -1, 1), np.clip(np.array(rotation, dtype=float), -1, 1), [grip]])

    def _grip_cmd(self):
        """Sticky gripper command: the gripper stays in its current state (closed or open) across
        every move/rotate/wait. Only `grasp` (-> closed) and `release` (-> open) toggle it, so a
        grasped object is NOT dropped when a later move starts."""
        return 1.0 if self._gripper_closed else -1.0

    def _is_success(self):
        """Task success, branching on cfg.task_type:
        - pick_and_place: red cube resting INSIDE the bin (xy within the interior, z between
          the floor and the rim — xyz ALL checked) AND the gripper has released it.
        - pick_up (default): cube lifted above the table AND near the end-effector (held)."""
        if cfg.task_type == "pick_and_place":
            return self._is_pick_and_place_success()
        cube_pos = self.env.sim.data.body_xpos[self.env.cube_body_id]
        if cube_pos[2] <= self._tz + 0.05:
            return False
        ee = self._ee_pos()
        return np.linalg.norm(ee - cube_pos) < 0.10

    def _is_pick_and_place_success(self):
        """Cube placed inside the bin and released. All three axes are checked:
          - xy: cube center within the bin INTERIOR (clear of the walls);
          - z : cube center between the bin floor and the rim (resting inside, neither held
                above nor sitting on top of the walls);
          - gripper OPEN (a release action was issued)."""
        env = self.env
        if getattr(env, "bin_body_id", None) is None or env.cube_body_id is None:
            return False
        cube_pos = env.sim.data.body_xpos[env.cube_body_id]
        bin_pos = env.sim.data.body_xpos[env.bin_body_id]
        bin_size = np.asarray(env.bin_size, dtype=float)
        wall = float(env.bin_wall_thickness)
        table_z = float(env.table_offset[2])
        # interior half-extents (clear of the walls) minus a small safety margin
        half_x = (bin_size[0] - 2.0 * wall) / 2.0 - 0.005
        half_y = (bin_size[1] - 2.0 * wall) / 2.0 - 0.005
        inside_xy = (abs(cube_pos[0] - bin_pos[0]) < half_x and
                     abs(cube_pos[1] - bin_pos[1]) < half_y)
        floor_top = table_z + wall            # interior floor surface (world z)
        rim_top = table_z + bin_size[2]       # top of the walls (world z)
        inside_z = (cube_pos[2] > floor_top - 0.02) and (cube_pos[2] < rim_top - 0.01)
        gripper_open = not self._gripper_closed
        return bool(inside_xy and inside_z and gripper_open)

    def _ee_over_bin(self):
        """True when the end-effector xy sits within the bin interior (the same interior test as
        _is_pick_and_place_success, applied to the EE rather than the cube). The release guard uses
        this so a pick_and_place `release` only fires over the bin — a premature planner release
        that would drop the cube on the table is refused and the gripper stays closed."""
        env = self.env
        if getattr(env, "bin_body_id", None) is None:
            return False
        ee = self._ee_pos()
        bin_pos = env.sim.data.body_xpos[env.bin_body_id]
        bin_size = np.asarray(env.bin_size, dtype=float)
        wall = float(env.bin_wall_thickness)
        half_x = (bin_size[0] - 2.0 * wall) / 2.0 - 0.005
        half_y = (bin_size[1] - 2.0 * wall) / 2.0 - 0.005
        return bool(abs(ee[0] - bin_pos[0]) < half_x and abs(ee[1] - bin_pos[1]) < half_y)

    def _target_over_bin(self, target_pos):
        """True when target_pos xy is over the bin FOOTPRINT (outer, including the walls) — used by
        the above_object rim-clamp to decide whether a carry-target must clear the rim. Broader than
        _ee_over_bin (which tests the interior) because the approach has to clear the wall too."""
        env = self.env
        if getattr(env, "bin_body_id", None) is None:
            return False
        bin_pos = env.sim.data.body_xpos[env.bin_body_id]
        bin_size = np.asarray(env.bin_size, dtype=float)
        half_x = bin_size[0] / 2.0
        half_y = bin_size[1] / 2.0
        return bool(abs(target_pos[0] - bin_pos[0]) < half_x
                    and abs(target_pos[1] - bin_pos[1]) < half_y)

    def _rim_top(self):
        """World-z of the top of the bin walls (table_z + bin_size[2]), or None if no bin."""
        if getattr(self.env, "bin_body_id", None) is None:
            return None
        return float(self.env.table_offset[2]) + float(np.asarray(self.env.bin_size)[2])

    def _bin_footprint(self):
        """(cx, cy, hx, hy) of the bin OUTER footprint (includes the walls), or None if no bin."""
        if getattr(self.env, "bin_body_id", None) is None:
            return None
        bin_pos = self.env.sim.data.body_xpos[self.env.bin_body_id]
        bin_size = np.asarray(self.env.bin_size, dtype=float)
        return (float(bin_pos[0]), float(bin_pos[1]), bin_size[0] / 2.0, bin_size[1] / 2.0)

    def _carry_crosses_bin(self, target_pos):
        """True when the gripper is CLOSED (carrying) in pick_and_place AND the EE is currently OUTSIDE
        the bin footprint but the straight-line xy path to target_pos enters it. That is the only case
        where a naive straight-line move would drive the held object THROUGH a bin wall: the
        above_object rim-clamp sets the endpoint above the rim but the diagonal PATH still dips under
        the near wall, and a move_by carry bypasses the clamp entirely. When the EE is already over the
        bin opening (an intended descend-to-place) this returns False so the drop-in is not blocked."""
        if cfg.task_type != "pick_and_place":
            return False
        if getattr(self.env, "bin_body_id", None) is None:
            return False
        if not self._gripper_closed:
            return False
        fp = self._bin_footprint()
        cx, cy, hx, hy = fp

        def _inside(x, y):
            return abs(x - cx) < hx and abs(y - cy) < hy

        ee = self._ee_pos()
        if _inside(ee[0], ee[1]):
            return False          # already over the opening: a descend-to-place is intended
        # sample the xy segment ee -> target; entering the footprint anywhere means a wall crossing
        for t in np.linspace(0.0, 1.0, 24):
            x = ee[0] + t * (target_pos[0] - ee[0])
            y = ee[1] + t * (target_pos[1] - ee[1])
            if _inside(x, y):
                return True
        return False

    def _all_views(self):
        """Render both stereo camera views (MuJoCo native orientation)."""
        return (render_rgb(self.env, camera=cfg.left_camera),
                render_rgb(self.env, camera=cfg.right_camera))

    def _step_env(self, action):
        """Wrap env.step so every action vector is recorded for deterministic replay."""
        a = np.asarray(action, dtype=float)
        self._actions.append(a.copy())
        return self.env.step(a)

    def _emit_frame(self, phase, plan=None, t3d=None, msg="", pause=False, bbox=None):
        """Record a frame event for replay (always) and render it live (only if on_frame set)."""
        self._frames.append({
            "step": self._step, "phase": phase, "plan": plan,
            "t3d": None if t3d is None else list(t3d),
            "msg": msg, "pause": pause,
            "bbox": None if bbox is None else {k: list(v) for k, v in bbox.items()},
        })
        if self.on_frame is not None:
            img_L, img_R = self._all_views()
            self.on_frame(img_L, img_R, phase, plan, t3d, msg, self._step, pause=pause, bbox=bbox)

    def _hold_and_capture(self, n=20):
        """After success, hold the EE still for n steps so the video shows the outcome clearly
        instead of cutting off at the instant of success. The gripper keeps its current state
        (closed for pick_up, open for pick_and_place) so a released cube stays put in the bin."""
        hold = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, self._grip_cmd()])
        cube_id = self.env.cube_body_id
        for _ in range(n):
            if self._step >= cfg.max_steps:
                break
            self._step += 1
            self._step_env(hold)
            z = float(self.env.sim.data.body_xpos[cube_id][2])
            self._emit_frame("SUCCESS holding", msg=f"cube held at z={z:.3f}")

    def get_recording(self):
        """Return recorded action stream + frame metadata for offline video replay."""
        return {"actions": np.array(self._actions, dtype=float), "frames": self._frames}

    def _record_history(self, action, result_str):
        frames = {cam: render_rgb(self.env, camera=cam) for cam in cfg.camera_names}
        desc = f'{action["type"]}(step={self._step},{result_str})'
        self._history_frames.append((frames, desc))
        if len(self._history_frames) > 2:
            self._history_frames.pop(0)

    def _build_history_desc(self):
        if not self._history_frames: return None
        cams = cfg.camera_names
        lines = [
            '\n[HISTORY] Execution snapshots at action completion (after current views, oldest first).',
            f'Each entry contains both stereo views ({" + ".join(cams)}).',
            f'{len(self._history_frames)} history entries:'
        ]
        for i, (_, desc) in enumerate(self._history_frames):
            lines.append(f'  History {i}: {desc}')
        return '\n'.join(lines)

    # ─── Shared move-to-position primitive ─────────────────

    def _move_to_position(self, target_pos, speed=0.5):
        """Move EE to target_pos using proportional control with plateau detection.

        CARRY-PATH SAFETY: when carrying an object in pick_and_place and the straight-line path to
        target_pos would cross the bin footprint, lift straight up to a cruise altitude above the rim
        FIRST, then move over. A direct diagonal from a low grasp height to a high bin target passes
        UNDER the rim at the near wall and clips it; the above_object rim-clamp only sets the endpoint,
        it does not route the path. Descending INTO the bin (EE already over the opening) is left
        untouched so an intentional place-descend is never blocked."""
        target_pos = np.asarray(target_pos, dtype=float)
        if self._carry_crosses_bin(target_pos):
            ee0 = self._ee_pos()
            cruise_z = self._rim_top() + 0.05      # clear the rim + held-object half-height
            lift_tgt = np.array([ee0[0], ee0[1], cruise_z])
            print(f"  [CARRY-LIFT] path crosses bin -> lift to cruise z={cruise_z:.3f} before moving over", flush=True)
            st = self._move_to_position(lift_tgt, speed)   # pure vertical; xy unchanged -> no re-trigger
            if st == "max_steps":
                return st
            target_pos = target_pos.copy()
            target_pos[2] = max(target_pos[2], cruise_z)   # hold cruise across the horizontal phase
        stagnant = 0
        prev_dist = np.linalg.norm(self._ee_pos() - target_pos)
        for _ in range(300):
            self._step += 1
            if self._step >= cfg.max_steps:
                return "max_steps"
            ee = self._ee_pos()
            dist = np.linalg.norm(ee - target_pos)
            if dist < 0.005:
                self._step_env(self._osc_action(np.zeros(3), self._grip_cmd())); break
            if abs(prev_dist - dist) < 0.001:
                stagnant += 1
                if stagnant > 15:
                    self._step_env(self._osc_action(np.zeros(3), self._grip_cmd())); break
            else:
                stagnant = 0
            prev_dist = dist
            d = target_pos - ee
            act = self._osc_action(d * speed, self._grip_cmd())
            self._step_env(act)
            if self._is_success():
                self._emit_frame("SUCCESS", msg=f"cube lifted to {self.env.sim.data.body_xpos[self.env.cube_body_id][2]:.3f}")
                return "task_done"
            self._emit_frame(f"Move #{self._action_count}",
                             t3d=target_pos, msg=f"dist={dist:.3f} stag={stagnant}")
        return "plateau" if stagnant > 15 else "reached" if dist < 0.005 else "max_steps"

    # ─── Action execution ───────────────────────────────────

    def _execute_action(self, action: dict, object_3d) -> str:
        atype = action["type"]
        speed = action.get("speed", 0.5)

        if atype == "move_to":
            if object_3d is None:
                print("  [WARN] move_to but no object_3d, skipping", flush=True)
                return "skipped"
            target = action.get("target", "object")
            ee_now = self._ee_pos()
            if target == "object":
                target_pos = object_3d.copy()
            elif target == "above_object":
                height = action.get("height", 0.05)
                target_pos = object_3d.copy()
                target_pos[2] += height
                # When carrying an object toward a position over the bin (pick_and_place), raise the
                # approach target above the bin rim so the held object clears the wall on the way in.
                # The planner's height (default 0.05) is sized for a small object and would place a
                # held cube at/below the rim, clipping the wall; clamp to rim_top + margin then.
                if (cfg.task_type == "pick_and_place"
                        and getattr(self.env, "bin_body_id", None) is not None
                        and self._gripper_closed
                        and self._target_over_bin(target_pos)):
                    rim_top = float(self.env.table_offset[2]) + float(np.asarray(self.env.bin_size)[2])
                    target_pos[2] = max(target_pos[2], rim_top + 0.04)
            else:
                print(f"  [WARN] unknown move_to target: {target}", flush=True)
                return "unknown_target"
            print(f"  [DEBUG] move_to {target}: EE=({ee_now[0]:.3f},{ee_now[1]:.3f},{ee_now[2]:.3f}) "
                  f"object_3d=({object_3d[0]:.3f},{object_3d[1]:.3f},{object_3d[2]:.3f}) "
                  f"target=({target_pos[0]:.3f},{target_pos[1]:.3f},{target_pos[2]:.3f}) "
                  f"delta=({target_pos[0]-ee_now[0]:.3f},{target_pos[1]-ee_now[1]:.3f},{target_pos[2]-ee_now[2]:.3f})", flush=True)
            return self._move_to_position(target_pos, speed)

        elif atype == "move_by":
            offset = action.get("offset", [0, 0, 0])
            if not isinstance(offset, list) or len(offset) != 3: offset = [0, 0, 0]
            ee_now = self._ee_pos()
            target_pos = ee_now + np.array(offset, dtype=float)
            print(f"  [DEBUG] move_by offset={offset}: EE=({ee_now[0]:.3f},{ee_now[1]:.3f},{ee_now[2]:.3f}) "
                  f"target=({target_pos[0]:.3f},{target_pos[1]:.3f},{target_pos[2]:.3f})", flush=True)
            return self._move_to_position(target_pos, speed)

        elif atype == "descend":
            speed = action.get("speed", 0.3)
            ee = self._ee_pos()
            target_pos = np.array([ee[0], ee[1], self._grasp_z])
            return self._move_to_position(target_pos, speed)

        elif atype == "rotate":
            rot = action.get("rotation", [0, 0, 0])
            if not isinstance(rot, list) or len(rot) != 3: rot = [0, 0, 0]
            max_rot = max(abs(v) for v in rot)
            steps = max(10, min(200, int(80 * max_rot / max(speed, 0.1))))
            for _ in range(steps):
                self._step += 1
                if self._step >= cfg.max_steps:
                    return "max_steps"
                self._step_env(self._osc_action(np.zeros(3), self._grip_cmd(), rot))
                if self._is_success():
                    self._emit_frame("SUCCESS", msg=f"cube lifted to {self.env.sim.data.body_xpos[self.env.cube_body_id][2]:.3f}")
                    return "task_done"
                self._emit_frame(f"Move #{self._action_count}: {atype}", msg=f"rot={rot}")
            return "done"

        elif atype == "grasp":
            for i in range(10):
                self._step += 1
                self._step_env(np.array([0, 0, 0, 0, 0, 0, 1.0]))
                if self._is_success():
                    self._gripper_closed = True
                    return "task_done"
                self._emit_frame(f"Move #{self._action_count}: {atype}", msg="grasping")
            self._gripper_closed = True
            return "done"

        elif atype == "release":
            # Release guard (pick_and_place only): refuse to open the gripper unless the EE is over
            # the bin interior. The planner sometimes emits `release` before reaching the bin;
            # without this guard the cube drops on the table and — because there is a single
            # object_3d — the loop can no longer target the cube to re-grasp it. Task types with no
            # bin are unaffected (the guard is a no-op). The blocked slot is consumed as a
            # closed-gripper hold so the action accounting stays consistent.
            if (cfg.task_type == "pick_and_place"
                    and getattr(self.env, "bin_body_id", None) is not None
                    and not self._ee_over_bin()):
                print("  [GUARD] release blocked: EE not over bin, keeping gripper closed", flush=True)
                for _ in range(10):
                    self._step += 1
                    if self._step >= cfg.max_steps:
                        break
                    self._step_env(self._osc_action(np.zeros(3), self._grip_cmd()))
                return "blocked"
            for _ in range(10):
                self._step += 1
                self._step_env(np.array([0, 0, 0, 0, 0, 0, -1.0]))
            self._gripper_closed = False
            self._object_held = False    # released: the held latch clears with the gripper
            return "done"

        elif atype == "wait":
            for _ in range(10):
                self._step += 1
                self._step_env(self._osc_action(np.zeros(3), self._grip_cmd()))
            return "done"
        return "unknown"

    # ─── Stereo detection + triangulation ───────────────────

    def _detect_and_project(self, task: str, target_desc: str = None) -> tuple:
        # Called only when the task needs an object (gated in run_closedloop).
        # Query priority: target_desc (from task analysis) > cfg.bbox_target > generic.
        if target_desc:
            queries = [target_desc]
            print(f"  [DETECT] using target='{target_desc}'", flush=True)
        elif cfg.bbox_target is not None:
            queries = [cfg.bbox_target]
            print(f"  [DETECT] using bbox_target='{cfg.bbox_target}'", flush=True)
        else:
            queries = ["object"]
            print("  [DETECT] no target specified, using generic 'object'", flush=True)

        # Single attempt: OWLv2 detection is deterministic (@torch.no_grad, argmax, no sampling),
        # so retrying within one round returns byte-identical boxes and can never turn a fail into a
        # success. Recovery from a failed re-detect is the caller's job, via the per-query cache.
        img_L, img_R = self._all_views()
        left_cands, right_cands = get_detector().detect_candidates(img_L, img_R, queries=queries)
        if not left_cands or not right_cands:
            print("  [BBOX] OWLv2 returned no detection", flush=True)
            return None, None
        # Stereo matching: each eye returns up to k candidate bboxes. Triangulate every L/R pair and
        # keep those whose rays meet within 5 cm (triangulate_rays' sanity threshold). Among the
        # consistent pairs pick the highest combined score. The per-eye argmax pair frequently points
        # at DIFFERENT objects for faint targets (e.g. a wood-floor / transparent-wall bin, where OWLv2
        # latches onto a faint wood patch that differs between viewpoints); requiring 3D consistency
        # plus score preference recovers the real target even when it was not #1 in both eyes.
        best = None  # (combined_score, left_bbox, right_bbox, score_left, score_right, point_3d)
        for lb, sl in left_cands:
            for rb, sr in right_cands:
                p3d = stereo_bbox_to_3d(lb, rb, cfg.camera_size, self.env)
                if p3d is not None and (best is None or (sl + sr) > best[0]):
                    best = (sl + sr, lb, rb, sl, sr, p3d)
        if best is None:
            print("  [BBOX] triangulation failed (no L/R candidate pair corresponds)", flush=True)
            return None, None
        _, left_bbox, right_bbox, score_left, score_right, object_3d = best
        print(f"  [OWLv2] L={left_bbox} (score={score_left:.3f})", flush=True)
        print(f"  [OWLv2] R={right_bbox} (score={score_right:.3f})", flush=True)
        print(f"  [OWLv2] 3D target=({object_3d[0]:.3f}, {object_3d[1]:.3f}, {object_3d[2]:.3f})", flush=True)
        self._target_cache[queries[0]] = object_3d.copy()  # remember this query's last pose
        bbox_msg = f"L={left_bbox} R={right_bbox}"
        bbox_data = {"left": left_bbox, "right": right_bbox}
        self._emit_frame("Stereo Bbox Detection", msg=bbox_msg, pause=True, bbox=bbox_data)
        return object_3d, (left_bbox, right_bbox)

    # ─── Main closed-loop ───────────────────────────────────

    def run_closedloop(self, task: str) -> TrialResult:
        n = cfg.plan_length_n
        max_st = cfg.max_steps
        self.planner._call_count = 0
        self._history_frames = []
        self._gripper_closed = False
        self._object_held = False   # per-trial: held latch resets so a prior trial's state does
                                    # not leak into the next
        self._target_cache = {}     # per-trial: query -> last-known 3D pose (reset so a random
                                    # cube spawn in trial N+1 never leaks trial N's cube pose)
        self._actions = []          # reset recorded action stream for this trial
        self._frames = []           # reset recorded frame metadata for this trial

        # --- Task analysis: does this task need object detection? (decided ONCE, up front) ---
        img_L, img_R = self._all_views()
        if cfg.bbox_mode == "never":
            self._needs_object, self._target_desc = False, None
        elif cfg.bbox_mode == "always":
            self._needs_object, self._target_desc = True, cfg.bbox_target
        else:  # "auto" - Qwen decides whether the task needs an object
            self._needs_object, self._target_desc = self.planner.analyze_task(img_L, img_R, task)
            # An explicit cfg.bbox_target overrides the query string whenever detection runs,
            # so pick-up tasks keep their exact OWLv2 query.
            if self._needs_object and cfg.bbox_target is not None:
                self._target_desc = cfg.bbox_target

        # --- Detection (only when the task needs an object) ---
        if self._needs_object:
            object_3d, bboxes = self._detect_and_project(task, self._target_desc)
            if object_3d is None:
                return TrialResult(False, 0, "no_bbox_detected")
        else:
            object_3d = None
            print("  [DETECT] skipped - task analysis decided no object detection needed", flush=True)

        # Initial plan
        completed_actions = []
        img_L, img_R = self._all_views()
        plan = self.planner.plan_initial(img_L, img_R, task, self._step, completed_actions,
                                         self._ee_pos(), n_actions=n, object_3d=object_3d,
                                         gripper_state="open", needs_object=self._needs_object)
        if plan is None:
            return TrialResult(False, 0, "initial_plan_failed")
        obj_msg = f"object at ({object_3d[0]:.3f}, {object_3d[1]:.3f}, {object_3d[2]:.3f})" if object_3d is not None else "no target detected"
        self._emit_frame("Initial VLM Plan", plan, object_3d, obj_msg, pause=True)

        action_buffer = list(plan["actions"])
        action_count = 0
        replan_streak = 0            # consecutive re-plan failures (capped to avoid hang)

        while self._step < max_st:
            if self._is_success():
                _succ_step = self._step          # true success step (before post-success hold)
                self._hold_and_capture(20)
                return TrialResult(True, _succ_step, None, plan)

            # Held latch: once the cube is observed clearly off the table with the gripper closed
            # (a grasp + lift succeeded), mark it held for the rest of the trial. This SURVIVES a
            # later descend-to-place (the cube is intentionally lowered again) so the planner/query
            # re-check is never fooled into thinking the grasp failed mid-place and re-targeting the
            # already-held object. Cleared only by an actual release or a trial reset.
            if self._gripper_closed and getattr(self.env, "cube_body_id", None) is not None:
                obj_z_held = float(self.env.sim.data.body_xpos[self.env.cube_body_id][2])
                if obj_z_held > self._tz + 0.06:
                    self._object_held = True

            if len(action_buffer) == 0:
                # Capture the current state once for this re-plan round (shared by query
                # re-check, stereo re-detection, and the VLM re-plan below).
                ee = self._ee_pos()
                img_L, img_R = self._all_views()
                grip_state = "closed" if self._gripper_closed else "open"
                # Report the TRUE held status, not the raw cube height. A descend-to-place
                # intentionally lowers a HELD cube back near the table, which used to read as
                # "still on table - grasp failed" and made the query re-check flip back to
                # re-locating the cube (the oscillation/timeout cause). The self._object_held
                # latch (set by a proven grasp+lift, cleared only by release) is authoritative:
                # if it is True the object IS secured regardless of the current cube z.
                if self._gripper_closed:
                    cube_id = getattr(self.env, "cube_body_id", None)
                    if self._object_held:
                        grip_state += (" (object SECURED — grasp succeeded and the object is held; "
                                       "do NOT re-grasp it; proceed to place/next step)")
                    elif cube_id is not None:
                        obj_z = float(self.env.sim.data.body_xpos[cube_id][2])
                        grip_state += (f" (gripper closed but object NOT confirmed held, cube at "
                                       f"z={obj_z:.3f} — the grasp may have missed; locate the object "
                                       f"to grasp it)")
                history_images = {}
                for i, (frames_dict, _) in enumerate(self._history_frames):
                    for cam_name, frame in frames_dict.items():
                        history_images[f"history_{i}_{cam_name}"] = frame
                history_desc = self._build_history_desc()

                # Re-detect (only if this task needs an object) and re-plan.
                if self._needs_object:
                    # Per-round query regeneration: when no fixed query is configured
                    # (bbox_target is None), ask the VLM which single object to locate next
                    # from the current state. A fixed cfg.bbox_target skips regeneration.
                    if cfg.bbox_target is None:
                        cur_query = self.planner.regenerate_query(
                            img_L, img_R, task, self._step, completed_actions, ee,
                            gripper_state=grip_state,
                            history_images=history_images, history_desc=history_desc)
                        if cur_query:
                            self._target_desc = cur_query
                    new_object_3d, new_bboxes = self._detect_and_project(task, self._target_desc)
                    if new_object_3d is not None:
                        object_3d = new_object_3d
                    else:
                        # Fall back to THIS query's last-known pose (per-query cache) — never a
                        # different object's stale coordinates. If this query was never detected,
                        # there is genuinely no target this round and the planner must wait/move_by.
                        cached = self._target_cache.get(self._target_desc)
                        if cached is not None:
                            object_3d = np.array(cached, dtype=float)
                            print(f"  [WARN] re-detect failed, using cached '{self._target_desc}' "
                                  f"at ({object_3d[0]:.3f}, {object_3d[1]:.3f}, {object_3d[2]:.3f})", flush=True)
                        else:
                            object_3d = None
                            print(f"  [WARN] re-detect failed, no cached position for '{self._target_desc}'", flush=True)

                _t0 = time.time()

                new_plan = self.planner.plan_closedloop(
                    img_L, img_R, task, self._step, completed_actions, ee, n,
                    object_3d, gripper_state=grip_state,
                    history_images=history_images, history_desc=history_desc,
                    needs_object=self._needs_object)
                _elapsed = time.time() - _t0

                if new_plan is not None and len(new_plan["actions"]) > 0:
                    print(f"    Got {len(new_plan['actions'])} actions in {_elapsed:.0f}s", flush=True)
                    action_buffer = list(new_plan["actions"])
                    action_count = 0
                    replan_streak = 0
                    self._emit_frame("Re-plan", new_plan, object_3d,
                                   f"{len(action_buffer)} actions ({_elapsed:.0f}s)", pause=True)
                else:
                    replan_streak += 1
                    if replan_streak >= 10:
                        print(f"  VLM re-plan failed {replan_streak}x in a row, giving up", flush=True)
                        return TrialResult(False, self._step, "replan_failed", plan)
                    print(f"  VLM re-plan failed ({_elapsed:.0f}s), retrying... ({replan_streak}/10)", flush=True)
                    continue

            # Execute next action
            action = action_buffer.pop(0)
            action_count += 1
            atype = action["type"]
            params = {k: v for k, v in action.items() if k != "type"}
            self._action_count = action_count

            self._emit_frame(f"Move #{action_count}: {atype}",
                             t3d=object_3d, msg=f"params={json.dumps(params)}")

            result_str = self._execute_action(action, object_3d)
            self._record_history(action, result_str)
            completed_actions.append(f"{atype}(step={self._step},{result_str})")
            if len(completed_actions) > 20:
                completed_actions = completed_actions[-20:]

        return TrialResult(self._is_success(), self._step,
                           "timeout" if not self._is_success() else None, plan)

---
## 9. Diagnostic Check

In [ ]:
def diagnostic_check():
    sep = "=" * 60
    print(sep)
    print("VLM STEREO DIAGNOSTIC CHECK")
    print(sep)

    print("\n--- 1. Configuration ---")
    print(f"  VLM model: {cfg.vlm_model_id}")
    print(f"  Stereo:    L={cfg.left_camera}, R={cfg.right_camera}, baseline={cfg.stereo_baseline}m")
    print(f"  Full 3D:   {not cfg.stereo_use_table_z}")
    print(f"  Size:      {cfg.camera_size}")

    print("\n--- 2. Environment & Stereo Cameras ---")
    env = make_env(cfg)
    for cam_name in cfg.camera_names:
        cid = env.sim.model.camera_name2id(cam_name)
        pos = env.sim.model.cam_pos[cid].copy()
        fov = float(env.sim.model.cam_fovy[cid])
        print(f"  {cam_name}: pos=({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f}), fov={fov:.1f}°")
    cube_pos = env.sim.data.body_xpos[env.cube_body_id].copy()
    print(f"  Cube pos: ({cube_pos[0]:.3f}, {cube_pos[1]:.3f}, {cube_pos[2]:.3f})")

    print("\n--- 3. Stereo Triangulation Accuracy (perfect bbox) ---")
    (K_L, pos_L, R_L), (K_R, pos_R, R_R), tz = get_stereo_params(env)
    def _proj(p3d, cp, cR, K):
        pc = cR.T @ (p3d - cp)
        fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]
        return cx - fx*pc[0]/pc[2], cy - fy*pc[1]/pc[2]  # standard image coords
    u_L, v_L = _proj(cube_pos, pos_L, R_L, K_L)
    u_R, v_R = _proj(cube_pos, pos_R, R_R, K_R)
    hs = 20 / cfg.camera_size
    synth_lb = [u_L/384 - hs, v_L/384 - hs, u_L/384 + hs, v_L/384 + hs]
    synth_rb = [u_R/384 - hs, v_R/384 - hs, u_R/384 + hs, v_R/384 + hs]
    perfect_3d = stereo_bbox_to_3d(synth_lb, synth_rb, cfg.camera_size, env)
    if perfect_3d is not None:
        xy_err = np.linalg.norm(perfect_3d[:2] - cube_pos[:2])
        z_err = abs(perfect_3d[2] - cube_pos[2])
        xyz_err = np.linalg.norm(perfect_3d - cube_pos)
        print(f"  Perfect-bbox: XY={xy_err*100:.2f}cm, Z={z_err*100:.2f}cm, 3D={xyz_err*100:.2f}cm")
    else:
        print("  Perfect-bbox triangulation FAILED")

    print("\n--- 4. Phase 1: OWLv2 stereo bbox detection ---")
    img_L = render_rgb(env, camera=cfg.left_camera)
    img_R = render_rgb(env, camera=cfg.right_camera)
    queries = [cfg.bbox_target] if cfg.bbox_target else ["red cube"]
    bbox_result = get_detector().detect(img_L, img_R, queries=queries)
    if bbox_result is None:
        print("  BBOX DETECTION FAILED")
        env.close()
        return
    left_bbox, right_bbox, bbox_reasoning = bbox_result
    print(f"  Reasoning: {bbox_reasoning[:120] if bbox_reasoning else 'none'}")
    print(f"  Left bbox:  {left_bbox}")
    print(f"  Right bbox: {right_bbox}")

    object_3d = None  # safe default
    if not isinstance(left_bbox, str) and not isinstance(right_bbox, str):
        object_3d = stereo_bbox_to_3d(left_bbox, right_bbox, cfg.camera_size, env)
        if object_3d is not None:
            xy_err = np.linalg.norm(object_3d[:2] - cube_pos[:2])
            z_err = abs(object_3d[2] - cube_pos[2])
            xyz_err = np.linalg.norm(object_3d - cube_pos)
            print(f"\n  Detected 3D: ({object_3d[0]:.3f}, {object_3d[1]:.3f}, {object_3d[2]:.3f})")
            print(f"  True cube:   ({cube_pos[0]:.3f}, {cube_pos[1]:.3f}, {cube_pos[2]:.3f})")
            print(f"  XY error:    {xy_err*100:.1f} cm")
            print(f"  Z error:     {z_err*100:.1f} cm")
            print(f"  3D error:    {xyz_err*100:.1f} cm")
        else:
            print("  3D TRIANGULATION FAILED")
    else:
        print("  (bbox detection returned string tag, skipping 3D)")
    
    planner = get_planner()

    print("\n--- 5. Phase 2: Qwen action plan ---")
    task = "Pick up the red cube precisely. Position above the cube, descend, grasp firmly, and lift to 8 cm above the table."
    eef = env.robots[0].eef_site_id
    ee_site = eef[list(eef.keys())[0]] if isinstance(eef, dict) else eef
    ee_pos_init = env.sim.data.site_xpos[ee_site].copy()
    plan = planner.plan_initial(img_L, img_R, task, 0, [], ee_pos_init,
                                n_actions=cfg.plan_length_n, object_3d=object_3d,
                                gripper_state="open")
    if plan is None:
        print("  VLM PLAN FAILED")
    else:
        print(f"  Actions: {len(plan['actions'])} primitives")
        for i, a in enumerate(plan["actions"]):
            print(f"    {i+1}. {a}")
    ee_pos = env.sim.data.site_xpos[ee_site].copy()
    print(f"\n  EE pos: ({ee_pos[0]:.3f}, {ee_pos[1]:.3f}, {ee_pos[2]:.3f})")
    env.close()
    print(f"\n{sep}\nDiagnostic complete.\n{sep}")

if torch.cuda.is_available():
    diagnostic_check()
else:
    print("Skip diagnostic: no GPU")

---
## 10. Video Demo

## Experiment Configuration

Configure experiment parameters before running the video demo (edit the cell below).

### General parameters

- **`task_type`**: Task category — it ALSO selects the **success criterion**: `"pick_up"` (default; cube lifted & held) or `"pick_and_place"` (grasp the cube, drop it in the bin, release). See *Success (task-dependent)* below.
- **`user_prompt`**: Natural language task description. Be detailed — this goes directly into the VLM prompt as `Task: "..."`. Include target object, action sequence, precision requirements, and target height.
- **`cfg.bbox_mode`**: `"auto"` = Qwen decides at the very start whether object detection is needed (recommended); `"always"` = force detection; `"never"` = skip detection entirely (free motion / dance / wave).
- **`cfg.bbox_target`**: OWLv2 query text override (e.g. `"red cube"`). Used **only when detection actually runs**; ignored when Qwen decides the task needs no object.
- **`cfg.stereo_baseline`**: Distance between left and right camera eyes (meters); wider = better depth accuracy.
- **`cfg.object_x_range` / `cfg.object_y_range`**: XY bounds (meters, table frame) used when an object's position is `"random"` AND it has no per-entry `"range"` (a `"range"` overrides these for that one object). They are also the box the placement sampler searches, so keep them wide enough for every object to be laid out without overlap.
- **`video_path`**: Output path for the annotated video.

### Scene / objects — `cfg.objects` (dict)

`cfg.objects` is a dict `{ <spec>: <pos>, ... }` that fully controls what is on the table.

- **`<spec>` (key)** — the FIRST entry becomes the **task object** (`env.cube` / success target). Each key is one of:
  1. `"cube"` — the recolorable cube (see `cfg.cube_color` below).
  2. A **built-in model name** (table below) — keeps its native look.
  3. A **path** to an external `.xml` or `.obj` file — loaded as a custom mesh object.
- **`<pos>` (value)** — one of:
  1. `"random"` — uniform within `object_x_range`/`object_y_range` (or within a per-entry `"range"` if given).
  2. an `(x, y)` tuple — fixed position (meters, table frame).
  3. a **dict** `{"pos": "random" | (x,y), "range": [(xmin,xmax),(ymin,ymax)], "color": <preset>, <object-kwarg>: ...}` — `"range"` restricts ONE object's random spawn box (e.g. keep the cube off the bin footprint); `"color"` sets the color; any other keys (e.g. `bin_size`) forward to that object's constructor. `"range"` and `"color"` are NOT forwarded to the constructor.

#### Built-in model names (`cfg.objects` keys)

| key | class | notes |
|-----|-------|-------|
| `cube` | `BoxObject` | **only** recolorable key (`cfg.cube_color`); default task object |
| `can` | `CanObject` | mesh — recommended pickable |
| `lemon` | `LemonObject` | mesh — recommended pickable |
| `bottle` | `BottleObject` | mesh — recommended pickable |
| `milk` | `MilkObject` | mesh — recommended pickable |
| `cereal` | `CerealObject` | mesh — recommended pickable |
| `bread` | `BreadObject` | mesh — recommended pickable |
| `ball` | `BallObject` | primitive |
| `cylinder` | `CylinderObject` | primitive |
| `hollow_cylinder` | `HollowCylinderObject` | primitive |
| `capsule` | `CapsuleObject` | primitive |
| `cone` | `ConeObject` | primitive |
| `hammer` | `HammerObject` | composite (large footprint) |
| `pot_with_handles` | `PotWithHandlesObject` | composite (large footprint) |
| `bin` | `Bin` | composite — open-top 4-walled container; size configurable (see below); **not in the default scene** |
| `hinged_box` | `HingedBoxObject` | composite (articulated) |
| `ratcheting_wrench` | `RatchetingWrenchObject` | composite |
| `round_nut` | `RoundNutObject` | composite |
| `square_nut` | `SquareNutObject` | composite |
| `plate_with_hole` | `PlateWithHoleObject` | composite |
| `door` | `DoorObject` | composite (articulated; may place oddly) |

> The food / packaged / primitive models are the most reliable for pick-up and OOD experiments. The composite tools come from other robosuite task environments and may have a large footprint or spawn off-center — fine as distractors, less so as the pick target.

### Cube color — `cfg.cube_color`

Applied **only** when a `"cube"` key is present; ignored for every other model (they keep their native color). Presets:
`red` · `blue` · `green` · `purple` · `yellow` · `white` · `black`

### Bin (container) — `cfg.objects` key `"bin"`

A four-walled, **open-top container** (robosuite `Bin`) — the **drop target** for the `"pick_and_place"` task (the current default scene places one at `(0.12, 0.0)`, size `(0.16, 0.20, 0.10)`), or a large distractor otherwise. Add it with the `"bin"` key.

It is a textured composite, so passing `"color"` disables its baked wood texture and the solid color shows. Size/shape are configured through the **per-entry dict** — these keys forward straight to the constructor:

- **`bin_size`**: `(x, y, z)` full size in meters — default `(0.3, 0.3, 0.15)` (large; mind the table footprint).
- **`wall_thickness`**: wall thickness in meters — default `0.01`.
- **`transparent_walls`**: `True` (default) → semi-translucent walls; `False` → solid walls.
- **`color`**: any preset from the list above (flips off the texture so the color shows); omit it for the native brown wood look.

```python
# Default-size brown translucent bin, fixed near the back of the table
cfg.objects = {"cube": "random", "bin": {"pos": (-0.05, 0.20)}}

# Custom-size solid-blue bin as a drop target
cfg.objects = {"cube": (-0.10, -0.05),
               "bin": {"pos": (-0.05, 0.18), "bin_size": (0.22, 0.22, 0.12),
                       "transparent_walls": False, "color": "blue"}}
```

### Success (task-dependent)

`VLMController._is_success` branches on `cfg.task_type`:

- **`pick_up`** (default): the cube is lifted `> table_z + 0.05 m` **and** the end-effector is within `0.10 m` of it (still held).
- **`pick_and_place`**: the cube rests **inside** the bin **and** the gripper has **released** it. All three axes are checked — xy within the bin interior (clear of the walls), z between the bin floor and the rim (so a cube held above the bin, or one sitting on the rim, does **not** count), plus the open-gripper condition.

The bin interior is computed from the live `env.bin_body_id` / `env.bin_size` / `env.bin_wall_thickness`, so resizing the bin via `bin_size`/`wall_thickness` keeps the test correct.

### Examples

```python
# pick_and_place (current default): red cube spawns at random on the robot side
# (kept OUT of the bin footprint via "range"), wooden bin in front as the drop target,
# plus distractors. success = cube inside the bin + gripper released.
cfg.objects = {
    "cube":   {"pos": "random", "color": "red", "range": [(-0.22, -0.06), (-0.12, 0.12)]},
    "bin":    {"pos": (0.12, 0.0), "bin_size": (0.16, 0.20, 0.10)},
    "can":    {"pos": (0.17, 0.17), "color": "silver"},
    "bottle": (-0.15, -0.24),
    "milk":   (0.15, -0.20),
}

# Default: red cube at a random safe position (= original Lift behavior)
cfg.objects = {"cube": "random"}; cfg.cube_color = "red"

# Fixed blue cube
cfg.objects = {"cube": (-0.10, 0.05)}; cfg.cube_color = "blue"

# Pick the purple cube with a lemon distractor nearby
cfg.objects = {"cube": (-0.10, 0.0), "lemon": (-0.10, 0.12)}; cfg.cube_color = "purple"

# Multiple objects (cube stays the task object = first entry)
cfg.objects = {"cube": (-0.05, 0.0), "can": (-0.12, 0.08), "bottle": (-0.12, -0.08)}

# External mesh file (color param is ignored)
cfg.objects = {"/abs/path/to/mug.xml": (-0.10, 0.0)}; cfg.cube_color = "red"
```

See `load_new_3Dmodel_experience.md` for the external `.xml`/`.obj` import workflow (mesh + material + required sites; `scale` units: cm = 0.01, mm = 0.001).

In [ ]:
# ============================================================# Experiment Config -- edit before running Cell 10# ============================================================task_type = "pick_and_place"cfg.task_type = task_typeuser_prompt = ("Pick up the red cube and place it into the wooden bin. "               "The wooden bin is an open-top, four-walled container; its walls are about 10 cm (0.10 m) tall, "               "so the bin rim sits roughly 10 cm above the table surface. "               "Follow these steps in order:\n"               "1. Move the gripper directly above the red cube and align it over the cube in the XY plane.\n"               "2. Descend vertically until the gripper contacts the cube.\n"               "3. Close the gripper to grasp the cube firmly.\n"               "4. Lift the grasped cube straight UP by at least 0.15 m (use move_by offset [0,0,0.15] or more). "               "This raises the cube HIGHER than the 10 cm bin wall - the cube MUST clear the wall before you "               "move it sideways over the bin.\n"               "5. Move the gripper over the wooden bin and hover directly above the bin's opening, still "               "higher than the rim.\n"               "6. Descend vertically so the cube drops down into the bin, below the rim.\n"               "7. Open the gripper to release the cube inside the bin.\n"               "The task is complete when the red cube rests inside the bin and the gripper has released it.")# --- Stereo settings ---cfg.bbox_mode = "auto"          # "auto" (Qwen decides if detection needed) | "always" | "never"cfg.stereo_baseline = 0.50       # wider = better depth accuracycfg.bbox_target = None          # None = Qwen regenerates the query each re-check round (used only when detection runs)# Global sampler ranges: wide enough for the bin (16x20 cm footprint) + distractors to be placed# without overlap. Per-object "range" below overrides these for the ACTUAL spawn position in# _reset_internal, so these only need to give the sampler room (they do NOT constrain a# "random" object that carries its own "range").cfg.object_x_range = (-0.25, 0.25)cfg.object_y_range = (-0.25, 0.25)cfg.objects = {                       # all objects live HERE; empty {} = empty table    "cube":   {"pos": "random", "color": "red",                  # task object; color set HERE               "range": [(-0.22, -0.08), (-0.12, 0.12)]},        # random spawn X,Y (m) -> kept on the                                                                 # robot side, DISJOINT from the bin                                                                 # footprint (bin x >= -0.02), so the cube                                                                 # never starts inside the bin    "bin":    {"pos": (0.06, 0.0),                              # drop target: closer to the arm side;               "bin_size": (0.16, 0.20, 0.10)},                  # 16x20x10 cm wooden bin, clear of                                                                 # the distractors (footprint x in                                                                 # [-0.02,0.14], y in [-0.10,0.10])    "can":    {"pos": (0.13, 0.21), "color": "silver"},         # distractor (silver, not red)    "bottle": (-0.15, -0.24),                                   # distractor    "milk":   ( 0.15, -0.20),                                   # distractor}    # value = (x,y) | "random" | {"pos":..,"range":[(xmin,xmax),(ymin,ymax)],"color":..,<ctor-kwargs>};     # first entry = task object# ---video_path = "output.mp4"cfg.max_steps = 250      # pick-and-place spans more actions than a plain lift# ============================================================print(f"Config: task_type={task_type}")print(f"  user_prompt='{user_prompt}'")print(f"  bbox_mode={cfg.bbox_mode}")print(f"  objects={cfg.objects}")print(f"  stereo_baseline={cfg.stereo_baseline}m")print(f"  video_path={video_path}")print(f"  max_steps={cfg.max_steps}")

In [ ]:
def record_video(task, video_path):
    cam = VideoCapture(fps=8)
    planner = get_planner()
    env = make_env(cfg)

    img_L = render_rgb(env, camera=cfg.left_camera)
    img_R = render_rgb(env, camera=cfg.right_camera)
    cam.snap(img_L, img_R, "Initial", None, None, f'Task: "{task}"')

    ctrl = VLMController(env, planner, on_frame=None)
    from tqdm.notebook import tqdm
    pbar = tqdm(total=cfg.max_steps, desc="Steps", unit="step")
    _orig_snap = cam.snap

    def _snap_with_pbar(left_img, right_img, phase="", plan=None, t3d=None, msg="",
                        step=0, pause=False, bbox=None):
        try:
            pbar.n = min(step, cfg.max_steps)
            pbar.refresh()
            _orig_snap(left_img, right_img, phase, plan, t3d, msg, step, pause, bbox=bbox)
        except Exception as e:
            print(f"  [FRAME] snap error: {e}", flush=True)

    ctrl.on_frame = _snap_with_pbar
    result = ctrl.run_closedloop(task)
    pbar.n = min(ctrl._step, cfg.max_steps)
    pbar.refresh()
    pbar.close()

    final_msg = "SUCCESS" if result.success else f"FAILED ({result.error})"
    img_L = render_rgb(env, camera=cfg.left_camera)
    img_R = render_rgb(env, camera=cfg.right_camera)
    cam.snap(img_L, img_R, final_msg, result.plan, None, f"Steps: {result.steps}", step=result.steps)

    env.close()
    print(f"  [RECORD] Frames captured: {len(cam.frames)}", flush=True)
    path = cam.save(video_path)
    return path, result


if torch.cuda.is_available():
    path, result = record_video(task=user_prompt, video_path=video_path)
    print(f"Video saved: {path}")
    print(f"Result: {'SUCCESS' if result.success else 'FAIL'} ({result.error or 'ok'})")
else:
    print("Skip video: no GPU")

---
## 11. Batch Evaluation

In [ ]:
def evaluate_vlm(n: int, planner, task=None,
                  video_dir="trial_videos") -> tuple:
    """Run n trials and render+save a video for EACH one directly during the run.

    Frames are captured LIVE through the controller's on_frame hook (a per-trial
    VideoCapture) as the closed-loop executes -- no separate record->replay step. Each
    trial's video is saved immediately as {video_dir}/seed{NNNN}_{success|fail}.mp4,
    tagged with the trial outcome.

    task defaults to the module-level user_prompt (Experiment Config cell), so the task
    requirement has ONE source of truth -- edit user_prompt there to change it everywhere."""
    if task is None:
        task = user_prompt
    label = "Closed-Loop VLM (Stereo)"
    sep = "=" * 40
    print(f"\n{sep}")
    print(f" {label} Evaluation ({n} trials)")
    print(f"{sep}")
    results = []
    success_seeds = []
    os.makedirs(video_dir, exist_ok=True)
    for trial in range(n):
        trial_sep = "-" * 50
        print(f"\n  {trial_sep}")
        print(f"  > Trial {trial+1}/{n}")
        print(f"  {trial_sep}")

        trial_seed = cfg.seed + trial
        env = make_env(cfg, seed=trial_seed)
        cam = VideoCapture(fps=8)
        # Live frame capture: every controller frame event routes into cam via on_frame.
        ctrl = VLMController(env, planner, on_frame=cam.snap)
        # Opening title frame (before any action is taken).
        img_L = render_rgb(env, camera=cfg.left_camera)
        img_R = render_rgb(env, camera=cfg.right_camera)
        cam.snap(img_L, img_R, "Initial", None, None, f'Task: "{task}"')

        result = ctrl.run_closedloop(task)
        results.append(result)

        # Closing frame summarizing the outcome, then persist the video immediately.
        tag = "success" if result.success else "fail"
        final_msg = "SUCCESS" if result.success else f"FAILED ({result.error})"
        img_L = render_rgb(env, camera=cfg.left_camera)
        img_R = render_rgb(env, camera=cfg.right_camera)
        cam.snap(img_L, img_R, final_msg, result.plan, None,
                 f"Steps: {result.steps}", step=result.steps)

        vid_path = os.path.join(video_dir, f"seed{trial_seed:04d}_{tag}.mp4")
        cam.save(vid_path)
        if result.success:
            success_seeds.append(trial_seed)
            print(f"  [OK] SUCCESS - seed={trial_seed} ({result.steps} steps) -> {vid_path}")
        else:
            print(f"  [FAIL] ({result.error or 'ok'}) - {result.steps} steps -> {vid_path}")
        print(f"  {trial_sep}")
        env.close()

    # Save results + success seeds for analyze_vlm / summary (no model needed to read these).
    save_data = {
        "closed_loop": results,
        "success_seeds": success_seeds,
        "task": task,
    }
    torch.save(save_data, "vlm_eval_results.pt")

    n_ok = len(success_seeds)
    print(f"\n  {n_ok}/{n} trials succeeded. Videos saved to {video_dir}/")
    if success_seeds:
        print(f"  Success seeds: {success_seeds}")
    else:
        print(f"  No successful trials.")
    return results, success_seeds


def summary(results, label):
    n, ok = len(results), sum(1 for r in results if r.success)
    errs = {}
    for r in results:
        k = r.error or "ok"
        errs[k] = errs.get(k, 0) + 1
    print(f"\n{'='*40}\n{label}: {ok}/{n} = {100*ok/n:.1f}%")
    for e, c in sorted(errs.items()):
        print(f"  {e}: {c}")
    print("=" * 40)
    return ok / n


if torch.cuda.is_available():
    # max_steps is set ONCE in the Experiment Config cell (single source of truth, =250 for
    # pick-and-place). Do NOT re-set it here -- a second value silently overrides the config and
    # caused the batch to time out at the old 200 while single runs used 250.
    n = min(cfg.n_trials, 5)
    print("Loading VLM planner (shared across all trials)...")
    planner = get_planner()

    import os as _os
    _log_path = "vlm_batch_log.txt"
    vlm_log_path = _log_path
    if _os.path.exists(_log_path):
        _os.remove(_log_path)

    results, success_seeds = evaluate_vlm(n, planner)
    vlm_log_path = None
    print(f"\nVLM log written to {_log_path}")

    rc = summary(results, "Closed-Loop VLM (Stereo)")
    print(f"Saved to vlm_eval_results.pt (includes {len(success_seeds)} success seeds). "
          f"All {n} trial videos are in trial_videos/ as seed{{NNNN}}_{{success|fail}}.mp4")
else:
    print("Skip: no GPU")

---
## Trial Videos

Videos are now rendered and saved **directly during Batch Evaluation** (live frame capture through the controller's `on_frame` hook as each trial runs — no separate record→replay step).

- Output: `trial_videos/seed{NNNN}_{success|fail}.mp4` — one file per trial, tagged with the outcome.
- `vlm_eval_results.pt` (written by Batch Evaluation) holds the results + success seeds for `analyze_vlm` / `summary`.

> The per-trial action+frame recording machinery (`VLMController.get_recording`) is still present in the controller cell. To re-enable the old deferred replay (render videos later without re-running the model), re-add a `torch.save(ctrl.get_recording(), ...)` line in `evaluate_vlm` and a replay function; it is intentionally not wired up now that videos are saved inline.

---
## Model Cleanup


In [ ]:
# Run this cell to unload the VLM + OWLv2 and free GPU memory
if '_vlm_planner' in globals() and globals()['_vlm_planner'] is not None:
    _vlm_planner.unload()
    _vlm_planner = None
if '_detector' in globals() and globals()['_detector'] is not None:
    globals()['_detector'].unload()
    globals()['_detector'] = None
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print('Models unloaded, GPU memory freed')



---
## 12. Failure Mode Analysis

In [ ]:
def analyze_vlm(path="vlm_eval_results.pt"):
    if not os.path.exists(path):
        print("No saved results. Run Cell 11 first.")
        return
    data = torch.load(path, weights_only=False)
    for name, key in [("Closed-Loop VLM", "closed_loop")]:
        results = data[key]
        summary(results, name)
        modes = {}
        for r in results:
            if r.error:
                modes[r.error] = modes.get(r.error, 0) + 1
        if modes:
            print("  Failure breakdown:")
            for m, c in sorted(modes.items()):
                print(f"    {m}: {c}/{len(results)} ({100*c//len(results)}%)")

analyze_vlm()



---
## Usage Notes

| Step | Action |
|------|--------|
| 1 | Enable **GPU T4 x2** in Kaggle Accelerator |
| 2 | Run cells **1 → 12** in order |
| 3 | First run downloads Qwen3-VL-4B (~8 GB) — approx 2-3 min |
| 4 | **Cell 10** generates annotated video; **Cell 11** runs batch evaluation |
| 5 | Increase cfg.n_trials to 50+ in Cell 4 for publishable results |

### Parameter Tuning

| Parameter | Effect |
|-----------|--------|
| stereo_baseline | Larger = better depth accuracy but narrower overlap |
| stereo_use_table_z | False = full stereo 3D (XYZ), True = snap Z to table height |
| plan_length_n | Higher = longer lookahead |
| lift_height | How high to lift the object above table |
| use_bbox | False = skip stereo detection, VLM uses move_by only |

### Architecture

```ascii
Left Eye + Right Eye RGB Images + Task Description
         |
    [OWLv2]  ← open-vocabulary object detection
         |     outputs left_bbox + right_bbox
         v
    Dual bbox → Stereo Triangulation → 3D Position (X, Y, Z)
         |
    [Qwen3-VL-4B]  ← action planning only (no bbox)
         |            outputs DSL action plan
         v
    DSL Action Plan (semantic: move_to, descend, move_by, grasp, ...)
         |
    Code resolves targets:
      move_to → object_3d  (code-computed, no VLM arithmetic)
      descend → vertical to table_z
      move_by → ee_pos + offset  (VLM provides simple offset)
         |
    OSC_POSE Controller → Robot Execution
         |
    (action buffer empty)
         |
    [Loop: Re-detect + Re-plan with updated 3D context]
```

### Detection (OWLv2)

| Parameter | Description | Default |
|-----------|-------------|---------|
| `detection_queries` | Text queries for OWLv2 open-vocabulary detection | `["cube", "object", "target"]` |
| OWLv2 model | `google/owlv2-base-patch16-ensemble` (0.6B params) | — |

### Action Types

| Action | Target Resolution | When to Use |
|--------|-------------------|-------------|
| `move_to` (object) | `target = object_3d` (code) | Approach detected target |
| `move_to` (above_object) | `target = object_3d + [0,0,h]` (code) | Position above target |
| `move_by` | `target = ee + offset` (VLM) | Simple relative moves (lift, shift) |
| `descend` | `target_z = table_z` (code) | Vertical descent until contact |
| `grasp` / `release` / `wait` / `rotate` | — | Gripper & orientation control |